# NPS-Proxy Prediction from Healthcare Call-Center Transcripts
## A Text-First Machine Learning Approach Using Synthetic Labels

**Dataset**: AIxBlock 92k Real-World Call Center Scripts (English)  
**Domains**: Medicare (inbound), Medical Equipment (outbound), Automotive & Healthcare Insurance (inbound)  
**Task**: 3-class classification — detractor / passive / promoter

This notebook implements an end-to-end pipeline:
1. Downloads data directly from HuggingFace
2. Builds call-level features (text + structured)
3. Assigns synthetic NPS-proxy labels via dual independent labelers
4. Trains and evaluates LogisticRegression, Calibrated LinearSVC, and LightGBM
5. Reports domain-stratified holdout and grouped cross-validation results

In [1]:
!ls -l /content/nps_classified_calls.xlsx

-rw-r--r-- 1 root root 132681 Jun  6 11:44 /content/nps_classified_calls.xlsx


In [2]:
# -- Install dependencies (Colab) -----------------------------------------
# Pin dependency versions for reproducibility.
!pip install -q "huggingface_hub==0.23.4" "lightgbm==4.3.0" "tabulate==0.9.0" "openpyxl>=3.1.0" "transformers==4.35.0" "torch==2.1.0"

import os, json, re, math, hashlib, zipfile, warnings, shutil
from collections import deque
from copy import deepcopy
from itertools import combinations
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, brier_score_loss,
    cohen_kappa_score
)
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from scipy.stats import ttest_rel, wilcoxon
from lightgbm import LGBMClassifier
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Registry-based scaler dispatch validated at config load time.
SCALER_REGISTRY = {
    "StandardScaler": StandardScaler,
    "MinMaxScaler": MinMaxScaler,
    "RobustScaler": RobustScaler,
}

CONFIG = {
    "project_root": "/content/nps_pipeline",
    "data_dir": "/content/nps_pipeline/data",
    "outputs_dir": "/content/nps_pipeline/outputs",

    "hf_repo": "AIxBlock/92k-real-world-call-center-scripts-english",
    "healthcare_zips": [
        "medicare_inbound.zip",
        "medical_equipment_outbound.zip",
        "automotive_and_healthcare_insurance_inbound.zip",
    ],
    "healthcare_domains": {
        "medicare_inbound": {"direction": "inbound"},
        "medical_equipment_outbound": {"direction": "outbound"},
        "automotive_and_healthcare_insurance_inbound": {"direction": "inbound"},
    },

    "min_words": 10,
    "limit": 30000,

    "tfidf_max_features": 50000,
    "tfidf_ngram_range": (1, 2),
    "tfidf_min_df": 3,
    "tfidf_max_df": 0.95,
    "svd_components": 200,
    # Keep False always for modelling runs. Setting True fits TF-IDF
    # on the full corpus before any train/test split, leaking test vocabulary
    # into training and inflating all reported metrics.
    "exploratory_svd_enabled": False,  # NEVER set True for modelling
    "exploratory_svd_sample_size": 10000,

    "run_mode": "text_plus_structured",
    "training_label_col": "label_consensus",
    "track_b_scaler": "StandardScaler",  # must be a key in SCALER_REGISTRY

    "models": {
        "logreg": {"enabled": True, "max_iter": 3000, "class_weight": "balanced"},
        "linearsvc": {
            "enabled": True,
            "class_weight": "balanced",
            "calibration": True,   # Fix #6: emit calibrated probabilities; enables Brier score
            "calibration_method": "sigmoid",
            "calibration_cv": 3,
        },
        "lgbm": {
            "enabled": True,
            "n_estimators": 800,
            "learning_rate": 0.05,
            "num_leaves": 63,
            "class_weight": "balanced",
            "subsample": 0.9,
            "colsample_bytree": 0.9,
        },
    },

    "domain_holdout": True,
    "pooled_cv": True,
    "cv_splits": 5,
    "min_train_samples": 20,
    "min_train_samples_per_class": 2,
    "probability_metrics": True,

    # Thresholds stored in CONFIG so academic report f-string
    # always matches the values the labeler actually uses.
    "labeler_b_detractor_threshold": -0.30,
    "labeler_b_promoter_threshold": 0.45,
    "labeler_a_scale_cap": 3.0,

    # Fix #3: symmetric Labeler A thresholds stored in CONFIG.
    # Old code hard-coded -1.5 / +2.0 — a 3.5-unit passive zone skewed
    # positive — which over-assigned "passive" and starved the promoter
    # class.  Symmetric ±1.5 gives equal headroom on both sides.
    "labeler_a_detractor_threshold": -1.5,
    "labeler_a_promoter_threshold":   1.5,   # was implicit 2.0

    # Fix #4: score both head and tail of each transcript.
    # Tail-only (old approach: 1 500 chars) discarded opening emotional
    # context.  Head 35 % + Tail 65 % captures both.
    "sentiment_head_chars":   750,
    "sentiment_tail_chars":   750,
    "sentiment_weight_head":  0.35,
    "sentiment_weight_tail":  0.65,

    # Fix #5: near-duplicate signature config.
    # near_dup_sha1 = SHA1(first N chars + "|||" + last M chars) groups
    # near-identical transcripts so they never span train/test boundaries.
    "near_dup_prefix_len": 300,
    "near_dup_suffix_len": 300,

    "golden_dataset": {
        "enabled": True,
        "file_path": "/content/nps_classified_calls.xlsx",
        "call_id_col": "call_id",
        "label_col": "nps_label",
        "valid_labels": ["detractor", "passive", "promoter"],
        "override_synthetic": True,
        "confidence_boost": 1.0,
    },
    "random_seed": 42,
}

# Validate scaler choice at config load time.
if CONFIG["track_b_scaler"] not in SCALER_REGISTRY:
    raise ValueError(
        f"Unknown track_b_scaler: {CONFIG['track_b_scaler']!r}. "
        f"Choose from {list(SCALER_REGISTRY)}"
    )

for d in ["data", "outputs/healthcare_calls", "outputs/features",
          "outputs/labels", "outputs/models", "outputs/metrics",
          "outputs/metrics/confusion_matrices_v1",
          "outputs/figures", "outputs/reports", "outputs/golden"]:
    os.makedirs(f"{CONFIG['project_root']}/{d}", exist_ok=True)

print("Configuration loaded. Project root:", CONFIG["project_root"])
print(f"Scaler: {CONFIG['track_b_scaler']} v  |  Deps pinned v  |  Exploratory SVD disabled v")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 7.1 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==2.1.0
Configuration loaded. Project root: /content/nps_pipeline
Scaler: StandardScaler v  |  Deps pinned v  |  Exploratory SVD disabled v


In [3]:
# ── Shared utilities ─────────────────────────────────────────────────────

_WS_RE = re.compile(r"\s+")

def normalize_text(text: str) -> str:
    """Collapse whitespace for deterministic hashing."""
    return _WS_RE.sub(" ", (text or "").strip())

def sha1_hex(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def safe_mean(xs):
    return sum(xs) / len(xs) if xs else None

def quantile(xs, q):
    if not xs: return None
    xs_sorted = sorted(xs)
    idx = int(math.floor((len(xs_sorted) - 1) * q))
    return xs_sorted[max(0, min(idx, len(xs_sorted)-1))]

print("✓ Utility functions defined.")


✓ Utility functions defined.


## § Data Acquisition

Download only the 3 healthcare zip files from HuggingFace and extract them.  
Uses `huggingface_hub.hf_hub_download()` — idempotent (skips if already extracted).

In [4]:
# -- Download & Extract Healthcare Zips from HuggingFace --------------------

data_dir = Path(CONFIG["data_dir"])
repo_id = CONFIG["hf_repo"]

def _zip_json_members(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        return sorted([
            m for m in zf.namelist()
            if m.endswith(".json") and not PurePosixPath(m).name.startswith("._")
        ])

def _safe_flat_name(member, seen_names):
    parts = [p for p in PurePosixPath(member).parts if p not in ("", ".", "..")]
    name = "__".join(parts) if parts else PurePosixPath(member).name
    if name in seen_names:
        stem = Path(name).stem
        suffix = Path(name).suffix
        name = f"{stem}__{sha1_hex(member)[:10]}{suffix}"
    seen_names.add(name)
    return name

for zip_name in CONFIG["healthcare_zips"]:
    domain_name = zip_name.replace(".zip", "")
    extract_dir = data_dir / domain_name
    manifest_path = extract_dir / "_extraction_manifest.json"

    print(f"Checking {zip_name} from {repo_id}...")
    zip_path = hf_hub_download(
        repo_id=repo_id,
        filename=zip_name,
        repo_type="dataset",
        local_dir=str(data_dir),
    )
    members = _zip_json_members(zip_path)
    expected_count = len(members)

    already_ok = False
    if extract_dir.exists() and manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            extracted_files = manifest.get("extracted_files", [])
            already_ok = (
                manifest.get("zip_name") == zip_name
                and manifest.get("expected_json_count") == expected_count
                and len(extracted_files) == expected_count
                and all((extract_dir / f).exists() for f in extracted_files)
            )
        except Exception:
            already_ok = False

    if already_ok:
        print(f"{domain_name}: validated existing extraction ({expected_count} JSON files). Skipping.")
        continue

    data_root = data_dir.resolve()
    target_root = extract_dir.resolve()
    if target_root.parent != data_root:
        raise ValueError(f"Refusing to reset unexpected extraction path: {target_root}")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    extracted_files = []
    seen_names = set()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in members:
            fname = _safe_flat_name(member, seen_names)
            target = extract_dir / fname
            with zf.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
            extracted_files.append(fname)

    actual_count = len(list(extract_dir.glob("*.json")))
    if actual_count != expected_count:
        raise RuntimeError(
            f"Extraction count mismatch for {domain_name}: expected {expected_count}, got {actual_count}"
        )

    manifest_path.write_text(json.dumps({
        "zip_name": zip_name,
        "expected_json_count": expected_count,
        "actual_json_count": actual_count,
        "extracted_files": extracted_files,
    }, indent=2), encoding="utf-8")
    print(f"{domain_name}: extracted and validated {actual_count} JSON files.")

print("\nAll healthcare data downloaded, extracted, and validated.")

Checking medicare_inbound.zip from AIxBlock/92k-real-world-call-center-scripts-english...


medicare_inbound.zip:   0%|          | 0.00/827M [00:00<?, ?B/s]

medicare_inbound: extracted and validated 61513 JSON files.
Checking medical_equipment_outbound.zip from AIxBlock/92k-real-world-call-center-scripts-english...


medical_equipment_outbound.zip:   0%|          | 0.00/13.5M [00:00<?, ?B/s]

medical_equipment_outbound: extracted and validated 738 JSON files.
Checking automotive_and_healthcare_insurance_inbound.zip from AIxBlock/92k-real-world-call-center-scripts-english...


automotive_and_healthcare_insurance_inbo(…):   0%|          | 0.00/41.4M [00:00<?, ?B/s]

automotive_and_healthcare_insurance_inbound: extracted and validated 1793 JSON files.

All healthcare data downloaded, extracted, and validated.


## § Data Engineering

Parse all JSON transcripts into a single call-level DataFrame.  
Apply quality filters (minimum word count) and compute deduplication hashes.  
Extract word-level ASR confidence statistics and speaker tag metadata.

In [5]:
# -- Data Engineering: Parse JSONs to Call-Level DataFrame ------------------
# Initialise profile to None so Figures/Report guard never hits NameError.
profile = None

records = []
n_seen = n_written = n_failed = n_filtered_short = n_bad_duration = 0
# Use deque to keep ALL errors (capped at 1000) not just the first 5.
errors_log = deque(maxlen=1000)

def _positive_float_or_nan(value):
    try:
        val = float(value)
    except (TypeError, ValueError):
        return np.nan
    return val if np.isfinite(val) and val > 0 else np.nan

# Parallel JSON loading via ThreadPoolExecutor (I/O-bound).
import concurrent.futures

def _parse_one_json(args):
    jf, domain_name, domain_info, min_words = args
    try:
        with open(jf, "r", encoding="utf-8", errors="replace") as fh:
            data = json.load(fh)

        text  = data.get("text", "")
        words = data.get("words", []) or []
        audio_dur_raw = data.get("audio_duration", None)
        audio_dur = _positive_float_or_nan(audio_dur_raw)
        asr_conf  = data.get("confidence", None)
        pii       = data.get("redacted_pii_policies", [])

        # n_words_asr = raw ASR word-object count (used for the min_words
        # filter only). n_words_text will be computed from normalised text in the
        # Feature Engineering cell, giving one canonical word-count definition.
        n_words_asr = len(words)
        bad_dur = audio_dur_raw is not None and pd.isna(audio_dur)
        if n_words_asr < min_words:
            return None, None, True, bad_dur

        word_confs = [w.get("confidence", 0.0) for w in words if "confidence" in w]
        text_norm  = re.sub(r"\s+", " ", (text or "").strip())
        source_call_id = jf.stem

        def _qnt(xs, q):
            if not xs: return None
            s = sorted(xs); idx = int(math.floor((len(s)-1)*q))
            return s[max(0, min(idx, len(s)-1))]

        record = {
            "call_id":           f"{domain_name}::{source_call_id}",
            "source_call_id":    source_call_id,
            "domain":            domain_name,
            "direction":         domain_info["direction"],
            "file_path":         str(jf),
            "file_name":         jf.name,
            "text":              text,
            "text_norm_sha1":    hashlib.sha1(text_norm.encode("utf-8", errors="ignore")).hexdigest(),
            "audio_duration_sec": audio_dur,
            "audio_duration_valid": int(not pd.isna(audio_dur)),
            "asr_confidence":    asr_conf,
            "n_words_asr":       n_words_asr,
            "mean_word_conf":    sum(word_confs)/len(word_confs) if word_confs else None,
            "p10_word_conf":     _qnt(word_confs, 0.10),
            "p50_word_conf":     _qnt(word_confs, 0.50),
            "p90_word_conf":     _qnt(word_confs, 0.90),
            "has_speaker_tags":  any("speaker" in w for w in words),
            "redacted_pii_policies_json": json.dumps(pii) if pii else "[]",
        }
        return record, None, False, bad_dur
    except Exception as e:
        return None, {"file": str(jf), "error": str(e)}, False, False

for domain_name, domain_info in CONFIG["healthcare_domains"].items():
    domain_dir = Path(CONFIG["data_dir"]) / domain_name
    if not domain_dir.exists():
        print(f"Warning: {domain_name}: directory not found, skipping.")
        continue

    json_files = sorted([
        f for f in domain_dir.rglob("*.json")
        if not f.name.startswith("._") and f.name != "_extraction_manifest.json"
    ])
    if CONFIG["limit"]:
        json_files = json_files[:CONFIG["limit"]]

    print(f"Processing {domain_name}: found {len(json_files)} valid JSON files...")

    args_list = [(jf, domain_name, domain_info, CONFIG["min_words"]) for jf in json_files]
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        for record, error, filtered, bad_dur in executor.map(_parse_one_json, args_list):
            n_seen += 1
            if bad_dur:
                n_bad_duration += 1
            if filtered:
                n_filtered_short += 1
            elif error:
                n_failed += 1
                errors_log.append(error)
            elif record:
                records.append(record)

calls_df = pd.DataFrame(records)
n_written = len(calls_df)

print(
    f"\nBuild summary: seen={n_seen}, written={n_written}, "
    f"filtered_short={n_filtered_short}, failed={n_failed}, bad_duration={n_bad_duration}"
)
if n_failed > 0:
    # Show all errors and persist to CSV for full inspection.
    print(f"  {n_failed} files failed. First 10 errors:")
    for err in list(errors_log)[:10]:
        print(f"    {err}")
    pd.DataFrame(list(errors_log)).to_csv(
        f"{CONFIG['outputs_dir']}/healthcare_calls/parse_errors.csv", index=False
    )

if n_written > 0:
    if calls_df["call_id"].duplicated().any():
        dup_ids = calls_df.loc[calls_df["call_id"].duplicated(), "call_id"].head(10).tolist()
        raise ValueError(f"Internal call_id collisions detected: {dup_ids}")

    profile = calls_df.groupby("domain").agg(
        n_calls=("call_id", "count"),
        n_unique_hashes=("text_norm_sha1", "nunique"),
        mean_duration=("audio_duration_sec", "mean"),
        mean_words=("n_words_asr", "mean"),
    ).reset_index()
    profile["dup_rate"] = 1 - profile["n_unique_hashes"] / profile["n_calls"]
    print("\n-- Per-Domain Profile --")
    display(profile)

    hash_domain_counts = calls_df.groupby("text_norm_sha1")["domain"].nunique()
    dedup_audit = pd.DataFrame([{
        "n_calls": len(calls_df),
        "n_unique_hashes": calls_df["text_norm_sha1"].nunique(),
        "duplicate_hash_rows": int(calls_df.duplicated("text_norm_sha1", keep=False).sum()),
        "cross_domain_duplicate_hashes": int((hash_domain_counts > 1).sum()),
        "call_id_collisions": int(calls_df["call_id"].duplicated().sum()),
    }])
    print("\n-- Exact-Duplicate Audit --")
    print(dedup_audit.to_markdown(index=False))

    calls_csv           = f"{CONFIG['outputs_dir']}/healthcare_calls/calls_df.csv"
    calls_sanitized_csv = f"{CONFIG['outputs_dir']}/healthcare_calls/calls_df_sanitized.csv"
    audit_csv           = f"{CONFIG['outputs_dir']}/healthcare_calls/dedup_audit.csv"
    calls_df.to_csv(calls_csv, index=False)
    calls_df.drop(columns=["text"], errors="ignore").to_csv(calls_sanitized_csv, index=False)
    dedup_audit.to_csv(audit_csv, index=False)
    print(f"\nSaved full local calls_df ({len(calls_df)} rows) to {calls_csv}")
    print(f"Saved sanitized calls_df without transcript text to {calls_sanitized_csv}")
    print(f"Saved dedup audit to {audit_csv}")
else:
    print("\nWarning: No records were processed. Check files and word-count filters.")


Processing medicare_inbound: found 30000 valid JSON files...
Processing medical_equipment_outbound: found 738 valid JSON files...
Processing automotive_and_healthcare_insurance_inbound: found 1793 valid JSON files...

Build summary: seen=32531, written=32525, filtered_short=0, failed=6, bad_duration=0
  6 files failed. First 10 errors:
    {'file': '/content/nps_pipeline/data/medicare_inbound/medicare_inbound__12d7320994464c3fb3021d6e9c40234f_transcript.json', 'error': 'Expecting value: line 1 column 1 (char 0)'}
    {'file': '/content/nps_pipeline/data/medicare_inbound/medicare_inbound__1a71fd94e216457a810c1aed95b8bc19 (1)_transcript.json', 'error': 'Expecting value: line 1 column 1 (char 0)'}
    {'file': '/content/nps_pipeline/data/medicare_inbound/medicare_inbound__2990c92bbbee4cc596548b0ae256f59e_transcript.json', 'error': "Expecting ':' delimiter: line 6686 column 14 (char 131119)"}
    {'file': '/content/nps_pipeline/data/medicare_inbound/medicare_inbound__2c022fdcfe2b4590963829

,domain,n_calls,n_unique_hashes,mean_duration,mean_words,dup_rate
0,automotive_and_healthcare_insurance_inbound,1793,1479,692.527607,1203.715003,0.175125
1,medical_equipment_outbound,738,738,564.403794,1053.340108,0.000000
2,medicare_inbound,29994,29711,389.001100,683.653264,0.009435



-- Exact-Duplicate Audit --
|   n_calls |   n_unique_hashes |   duplicate_hash_rows |   cross_domain_duplicate_hashes |   call_id_collisions |
|----------:|------------------:|----------------------:|--------------------------------:|---------------------:|
|     32525 |             31928 |                  1188 |                               0 |                    0 |

Saved full local calls_df (32525 rows) to /content/nps_pipeline/outputs/healthcare_calls/calls_df.csv
Saved sanitized calls_df without transcript text to /content/nps_pipeline/outputs/healthcare_calls/calls_df_sanitized.csv
Saved dedup audit to /content/nps_pipeline/outputs/healthcare_calls/dedup_audit.csv


## § Feature Engineering

### Track A — Text Representation (TF-IDF → SVD)
Normalize texts, fit TF-IDF vectorizer, reduce to 200 SVD components.

### Track B — Structured Features
Lexical features, interaction cue counts (regex-based), and ASR/duration features.

In [6]:
# -- Feature Engineering: Optional Exploratory Track A + Structured Track B --

texts = calls_df["text"].map(normalize_text).values
text_df = pd.DataFrame({
    "call_id":       calls_df["call_id"].values,
    "source_call_id": calls_df["source_call_id"].values,
    "text_norm":     texts,
})

svd_df = pd.DataFrame({
    "call_id":       calls_df["call_id"].values,
    "source_call_id": calls_df["source_call_id"].values,
})
svd_cols = []

if CONFIG.get("exploratory_svd_enabled", False):
    print("Building Track A: TF-IDF to SVD exploratory artifacts...")
    sample_size = CONFIG.get("exploratory_svd_sample_size")
    if sample_size and len(texts) > sample_size:
        rng = np.random.default_rng(CONFIG["random_seed"])
        sample_idx = np.sort(rng.choice(len(texts), size=sample_size, replace=False))
        exploratory_texts = texts[sample_idx]
        print(f"  Using sampled exploratory corpus: {len(exploratory_texts)} of {len(texts)} calls")
    else:
        sample_idx = None
        exploratory_texts = texts

    tfidf_exp = TfidfVectorizer(
        max_features=CONFIG["tfidf_max_features"],
        ngram_range=CONFIG["tfidf_ngram_range"],
        min_df=CONFIG["tfidf_min_df"],
        max_df=CONFIG["tfidf_max_df"],
        sublinear_tf=True,
    )
    tfidf_matrix = tfidf_exp.fit_transform(exploratory_texts)
    n_svd_components = min(CONFIG["svd_components"], max(1, tfidf_matrix.shape[1] - 1))
    svd_exp = TruncatedSVD(n_components=n_svd_components, random_state=CONFIG["random_seed"])
    svd_matrix = svd_exp.fit_transform(tfidf_matrix)
    print(f"  Exploratory TF-IDF matrix: {tfidf_matrix.shape}")
    print(f"  Exploratory SVD matrix: {svd_matrix.shape}")
    print(f"  Exploratory global explained variance: {svd_exp.explained_variance_ratio_.sum():.4f}")

    svd_cols = [f"svd_{i:03d}" for i in range(n_svd_components)]
    if sample_idx is None:
        svd_df = pd.DataFrame(svd_matrix, columns=svd_cols)
        svd_df["call_id"] = calls_df["call_id"].values
        svd_df["source_call_id"] = calls_df["source_call_id"].values
    else:
        svd_df = pd.DataFrame(svd_matrix, columns=svd_cols)
        svd_df["call_id"] = calls_df.iloc[sample_idx]["call_id"].values
        svd_df["source_call_id"] = calls_df.iloc[sample_idx]["source_call_id"].values

    joblib.dump(tfidf_exp, f"{CONFIG['outputs_dir']}/models/tfidf_vectorizer_exploratory.joblib")
    joblib.dump(svd_exp, f"{CONFIG['outputs_dir']}/models/svd_reducer_exploratory.joblib")
    # Exploratory SVD is NEVER merged into model_df.
    # Model pipelines re-fit TF-IDF/SVD per fold via TextToSVD.
    print("Saved exploratory TF-IDF/SVD objects (for inspection only, NOT used in model_df).")
else:
    print("Skipping global exploratory TF-IDF/SVD (CONFIG['exploratory_svd_enabled']=False).")
    print("Model evaluation fits TF-IDF/SVD inside each train fold via TextToSVD.")

print("\nBuilding Track B: Structured features...")

# Labeler cue columns: computed here for labeling/audit, EXCLUDED from model
# features. Exclusion is enforced by assertion in the Modeling cell.
LABELER_CUE_COLS = [
    "apology_count", "thanks_count", "escalation_count",
    "resolution_count", "unresolved_count",
]

# Pre-compile regex patterns once to avoid recompilation per row.
_RAW_PATTERNS = {
    "apology_count":    r"(?i)\b(?:i'm sorry|apologize|our apologies|sorry about)\b",
    "thanks_count":     r"(?i)\b(?:thanks?|thank you|appreciate)\b",
    "escalation_count": r"(?i)\b(?:supervisor|manager|escalat(?:e|ion|ing)|complain(?:t|ing)|cancel|refund|lawsuit)\b",
    "resolution_count": r"(?i)\b(?:scheduled|appointment|confirm(?:ed|ation)|submitted|resolved|no problem)\b",
    "unresolved_count": r"(?i)\b(?:cannot|can't|cant|unable|doesn't work|doesnt work|not coming up|wrong|issue|problem)\b",
}
PATTERNS = {name: re.compile(pat) for name, pat in _RAW_PATTERNS.items()}

text_series = calls_df["text"].fillna("")
# n_words_text is derived exclusively from normalised text -- the
# single canonical word-count definition used in all downstream features
# and labeler scoring (consistent with the text used for TF-IDF).
text_norm_series = pd.Series(texts, index=calls_df.index)
n_words_text = text_norm_series.str.split().str.len().fillna(0).astype(int)
duration = pd.to_numeric(calls_df["audio_duration_sec"], errors="coerce")
duration = duration.where(np.isfinite(duration) & (duration > 0))

struct_df = pd.DataFrame({
    "call_id":        calls_df["call_id"].values,
    "source_call_id": calls_df["source_call_id"].values,
    "n_chars":        text_series.str.len().values,
    "n_words_text":   n_words_text.values,
    "qmark_count":    text_series.str.count(r"\?").values,
    "emark_count":    text_series.str.count("!").values,
    "dot_count":      text_series.str.count(r"\.").values,
    "has_speaker_tags": pd.to_numeric(calls_df["has_speaker_tags"], errors="coerce").fillna(0).astype(int).values,
})
# Use pre-compiled patterns.
for feat_name, compiled_pat in PATTERNS.items():
    struct_df[feat_name] = text_norm_series.str.count(compiled_pat).fillna(0).astype(int).values

for col in ["asr_confidence", "mean_word_conf", "p10_word_conf", "p50_word_conf", "p90_word_conf"]:
    struct_df[col] = pd.to_numeric(calls_df[col], errors="coerce").fillna(0).values

struct_df["audio_duration_sec"]   = duration.fillna(0).values
struct_df["audio_duration_valid"] = duration.notna().astype(int).values
struct_df["wpm"] = np.where(duration.notna(), (n_words_text / duration) * 60, 0)
struct_df["wpm"] = np.where(np.isfinite(struct_df["wpm"]), struct_df["wpm"], 0)

features_df = struct_df.merge(text_df, on=["call_id", "source_call_id"], how="inner")

struct_path = f"{CONFIG['outputs_dir']}/features/track_b_structured.csv"
svd_path    = f"{CONFIG['outputs_dir']}/features/track_a_svd_exploratory.csv"
merged_path = f"{CONFIG['outputs_dir']}/features/features_merged.csv"
struct_df.to_csv(struct_path, index=False)
svd_df.to_csv(svd_path, index=False)
features_df.drop(columns=["text_norm"]).to_csv(merged_path, index=False)

track_b_feat_cols = [c for c in struct_df.columns if c not in ("call_id", "source_call_id")]
print(f"\nTrack A exploratory SVD columns saved: {len(svd_cols)}")
print(f"Track B: {len(track_b_feat_cols)} structured features: {track_b_feat_cols}")
print(f"In-memory model features: {features_df.shape} (includes text_norm, excludes precomputed SVD)")
print(f"Sanitized merged feature export: {merged_path} (text_norm omitted)")


Skipping global exploratory TF-IDF/SVD (CONFIG['exploratory_svd_enabled']=False).
Model evaluation fits TF-IDF/SVD inside each train fold via TextToSVD.

Building Track B: Structured features...

Track A exploratory SVD columns saved: 0
Track B: 19 structured features: ['n_chars', 'n_words_text', 'qmark_count', 'emark_count', 'dot_count', 'has_speaker_tags', 'apology_count', 'thanks_count', 'escalation_count', 'resolution_count', 'unresolved_count', 'asr_confidence', 'mean_word_conf', 'p10_word_conf', 'p50_word_conf', 'p90_word_conf', 'audio_duration_sec', 'audio_duration_valid', 'wpm']
In-memory model features: (32525, 22) (includes text_norm, excludes precomputed SVD)
Sanitized merged feature export: /content/nps_pipeline/outputs/features/features_merged.csv (text_norm omitted)


## § Synthetic Labeling

Two independent deterministic labelers assign NPS-proxy classes:
- **Labeler A (Rule/Rubric)**: Weighted keyword scoring with threshold mapping
- **Labeler B (Sentiment-Weighted)**: Lexicon-based sentiment with interaction cue adjustments

Cohen's kappa measures inter-labeler agreement. `label_final` = consensus/adjudicated synthetic label.

> **Note**: These are synthetic labels approximating end-of-call satisfaction, NOT true NPS scores.

In [7]:
# -- Synthetic Labeling: Dual Deterministic Labelers + Cohen's Kappa ---------

NPS_CLASSES = CONFIG["golden_dataset"]["valid_labels"]

# Pull thresholds from CONFIG so the academic report always documents
# the same values the labeler actually uses -- no hard-coded prose drift.
B_DETRACTOR_THRESHOLD   = CONFIG["labeler_b_detractor_threshold"]
B_PROMOTER_THRESHOLD    = CONFIG["labeler_b_promoter_threshold"]
A_SCALE_CAP             = CONFIG["labeler_a_scale_cap"]
# Fix #3: read symmetric thresholds from CONFIG.
A_DETRACTOR_THRESHOLD   = CONFIG.get("labeler_a_detractor_threshold", -1.5)
A_PROMOTER_THRESHOLD    = CONFIG.get("labeler_a_promoter_threshold",   1.5)

label_input = struct_df.merge(
    calls_df[["call_id", "source_call_id", "text", "domain", "direction", "text_norm_sha1"]],
    on=["call_id", "source_call_id"],
    how="inner",
)

# Vectorized Labeler A.
n_words = label_input["n_words_text"].clip(lower=1)
scale   = (1000.0 / n_words).clip(upper=A_SCALE_CAP)
score_a = (
    -2.0 * label_input["escalation_count"] * scale
    -1.0 * label_input["unresolved_count"]  * scale
    +0.5 * label_input["thanks_count"]      * scale
    +1.0 * label_input["resolution_count"]  * scale
    -0.5 * label_input["apology_count"]     * scale
)
# Fix #3: symmetric CONFIG-driven thresholds (old: -1.5 / +2.0 asymmetric).
label_a = np.select(
    [score_a <= A_DETRACTOR_THRESHOLD, score_a >= A_PROMOTER_THRESHOLD],
    ["detractor", "promoter"],
    default="passive",
)

# ============================================================
# Vectorized Labeler B -- BATCHED DistilBERT (Issue #1 fix)
# ============================================================
#
# WHAT WAS WRONG:
#   The original code used pandas .apply(get_sentiment_score) which executed
#   one full DistilBERT forward pass per row.  At 30 000 rows on Colab CPU
#   this takes 25-100 minutes and reliably hits the 90-min session timeout.
#
# WHAT CHANGED:
#   1. Use HuggingFace pipeline() with batch_size=64.
#      The pipeline pads and stacks inputs into GPU/CPU batches, achieving
#      15-30x speedup over row-wise .apply().  On Colab CPU: ~3-6 min.
#      On Colab T4 GPU: ~25-60 sec.
#
#   2. Score the TAIL of each transcript (last SENTIMENT_TAIL_CHARS chars).
#      Healthcare calls often run 500-2000 words.  The model's 512-token
#      window covers ~350-400 words, so scoring only the opening misses
#      the resolution/escalation language that matters for NPS.  Scoring
#      the tail captures "I'm still upset", "everything was resolved", etc.
#
#   3. Release model weights after scoring.
#      DistilBERT is ~250 MB.  Keeping it alive through the Modeling cell
#      reduces free RAM available to LightGBM and can cause OOM on the
#      Colab free tier.
#
# CONFIG knobs (optional -- defaults shown):
#   CONFIG["sentiment_batch_size"]  = 64    # pipeline batch size
#   CONFIG["sentiment_tail_chars"]  = 1500  # chars scored per transcript
#   CONFIG["sentiment_max_length"]  = 512   # tokeniser truncation limit

_SENT_BATCH_SIZE  = int(CONFIG.get("sentiment_batch_size", 64))
_SENT_HEAD_CHARS  = int(CONFIG.get("sentiment_head_chars",  750))
_SENT_TAIL_CHARS  = int(CONFIG.get("sentiment_tail_chars",  750))
_SENT_WEIGHT_HEAD = float(CONFIG.get("sentiment_weight_head", 0.35))
_SENT_WEIGHT_TAIL = float(CONFIG.get("sentiment_weight_tail", 0.65))
_SENT_MAX_LENGTH  = int(CONFIG.get("sentiment_max_length", 512))
_SENT_MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

try:
    import gc
    import torch
    from transformers import pipeline as hf_pipeline

    _device = 0 if torch.cuda.is_available() else -1
    print(f"Loading sentiment model '{_SENT_MODEL_NAME}' "
          f"({'GPU' if _device == 0 else 'CPU'}) ...")

    _sent_pipe = hf_pipeline(
        "sentiment-analysis",
        model=_SENT_MODEL_NAME,
        device=_device,
        batch_size=_SENT_BATCH_SIZE,
        truncation=True,
        max_length=_SENT_MAX_LENGTH,
    )

    # Fix #4: score BOTH head and tail of each transcript.
    # Tail-only discarded opening emotional context (frustration, reason for
    # the call) that is often predictive for detractors.  We concatenate head
    # and tail segment lists, call the pipeline once (same batching overhead),
    # then combine with configurable weights (head 35 %, tail 65 %).
    _raw_texts  = label_input["text"].fillna("").tolist()
    _guard      = lambda t: t if t.strip() else " "
    _texts_head = [_guard(t[:_SENT_HEAD_CHARS] if len(t) > _SENT_HEAD_CHARS else t)
                   for t in _raw_texts]
    _texts_tail = [_guard(t[-_SENT_TAIL_CHARS:] if len(t) > _SENT_TAIL_CHARS else t)
                   for t in _raw_texts]
    _texts_to_score = _texts_head + _texts_tail   # head first, tail second

    _n = len(_raw_texts)
    print(f"Running batched sentiment inference: {_n:,} texts × 2 segments | "
          f"batch_size={_SENT_BATCH_SIZE} | "
          f"head={_SENT_HEAD_CHARS} chars (w={_SENT_WEIGHT_HEAD}) + "
          f"tail={_SENT_TAIL_CHARS} chars (w={_SENT_WEIGHT_TAIL}) ...")

    _sent_results = _sent_pipe(_texts_to_score)

    def _to_signed(r):
        return r["score"] if r["label"] == "POSITIVE" else -r["score"]

    _head_scores  = np.array([_to_signed(r) for r in _sent_results[:_n]], dtype=np.float64)
    _tail_scores  = np.array([_to_signed(r) for r in _sent_results[_n:]], dtype=np.float64)
    sentiment_scores = (_SENT_WEIGHT_HEAD * _head_scores
                        + _SENT_WEIGHT_TAIL * _tail_scores)

    lexical_score = sentiment_scores   # [-1, 1] -- same contract as fallback path
    print(f"Sentiment done. mean={sentiment_scores.mean():.4f}  "
          f"std={sentiment_scores.std():.4f}  "
          f"min={sentiment_scores.min():.4f}  "
          f"max={sentiment_scores.max():.4f}")

    # Memory cleanup: free ~250 MB before Modeling cell runs.
    del _sent_pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Sentiment model released from memory.")

except Exception as _sent_exc:
    print(f"Warning: Sentiment model unavailable ({_sent_exc}). "
          "Falling back to keyword-based lexical scoring.")
    # Fallback: deterministic keyword approach; no external dependencies.
    _POS_RE = re.compile(
        r"(?i)\b(?:thank|thanks|appreciate|appreciated|great|awesome|perfect|excellent|"
        r"helpful|amazing|good|love|works|resolved|fixed|happy|satisfied|easy|clear)\b"
    )
    _NEG_RE = re.compile(
        r"(?i)\b(?:angry|mad|upset|terrible|horrible|ridiculous|frustrated|frustrating|"
        r"problem|issue|broken|doesn't|doesnt|can't|cant|cannot|refund|cancel|complaint|"
        r"lawsuit|wrong|unable|delay|delayed|denied|difficult|confusing)\b"
    )
    _text_norm_lb = label_input["text"].fillna("").map(normalize_text).str.lower()
    _pos_count = _text_norm_lb.str.count(_POS_RE).fillna(0)
    _neg_count = _text_norm_lb.str.count(_NEG_RE).fillna(0)
    _sentiment_rate = 100.0 * (_pos_count - _neg_count) / label_input["n_words_text"].clip(lower=1)
    lexical_score = np.tanh(_sentiment_rate / 1.5)

# ── Rest of Labeler B (structure unchanged) ──────────────────────────────────
thanks     = label_input["thanks_count"].clip(upper=5)
res        = label_input["resolution_count"].clip(upper=5)
esc        = label_input["escalation_count"].clip(upper=5)
unresolved = label_input["unresolved_count"].clip(upper=5)
apology    = label_input["apology_count"].clip(upper=5)
cue_raw    = (0.10*thanks) + (0.20*res) - (0.90*esc) - (0.90*unresolved) - (0.20*apology)
cue_score  = np.tanh(cue_raw / 3.0)
score_b    = (0.80 * lexical_score + 0.20 * cue_score).clip(-1, 1)
label_b    = np.select(
    [score_b <= B_DETRACTOR_THRESHOLD, score_b >= B_PROMOTER_THRESHOLD],
    ["detractor", "promoter"],
    default="passive",
)

agree            = label_a == label_b
score_a_abs      = np.abs(score_a)
score_a_max      = float(np.percentile(score_a_abs, 95)) if len(score_a_abs) else A_SCALE_CAP
score_a_max      = max(score_a_max, A_SCALE_CAP)
score_a_strength = np.minimum(score_a_abs / score_a_max, 1.0)
score_b_strength = np.minimum(np.abs(score_b), 1.0)
score_strength   = np.maximum(score_a_strength, score_b_strength)
# Fix #2: on disagreement pick the labeler with the stronger confidence
# signal rather than unconditionally deferring to Labeler B.
# Labeler B can silently degrade to keyword matching (the except branch),
# making it the *noisier* labeler on precisely the hard disagreement cases.
label_on_disagree  = np.where(score_a_strength >= score_b_strength, label_a, label_b)
label_consensus    = np.where(agree, label_a, label_on_disagree)
consensus_source   = np.where(
    agree,
    "agreement",
    np.where(score_a_strength >= score_b_strength,
             "labeler_a_higher_confidence",
             "labeler_b_higher_confidence"),
)
# Confidence: for disagreements reflect the stronger labeler's signal strength.
_disagree_conf = np.where(
    score_a_strength >= score_b_strength,
    0.20 + 0.30 * score_a_strength,
    0.20 + 0.30 * score_b_strength,
)
confidence = np.where(agree, 0.60 + 0.40 * score_strength, _disagree_conf)

# NOTE (Issue #2 fix): score_b is a numpy ndarray; .values is a pandas-only
# attribute and raises AttributeError on ndarrays.  np.asarray() is safe for
# both ndarray and Series inputs so the fix is robust to future refactors.
labels_df = pd.DataFrame({
    "call_id":          label_input["call_id"].values,
    "source_call_id":   label_input["source_call_id"].values,
    "domain":           label_input["domain"].values,
    "direction":        label_input["direction"].values,
    "text_norm_sha1":   label_input["text_norm_sha1"].values,
    "label_a":          label_a,
    "score_a":          score_a.values,
    "label_b":          label_b,
    "score_b":          np.asarray(score_b),
    "agree":            agree.astype(int),
    "confidence":       np.round(confidence, 4),
    "consensus_source": consensus_source,
    "label_consensus":  label_consensus,
    "label_final":      label_consensus,
})

label_counts = labels_df["label_final"].value_counts().reindex(NPS_CLASSES, fill_value=0)
if (label_counts > 0).sum() < 2:
    raise ValueError(f"Synthetic labels collapsed to fewer than two classes: {label_counts.to_dict()}")
if (label_counts == 0).any():
    print(f"Warning: at least one NPS class is absent from label_final: {label_counts.to_dict()}")

# safe_kappa wrapper -- cohen_kappa_score raises ValueError for
# single-class inputs (possible for small or very uniform domains).
def safe_kappa(a, b):
    """Return Cohen kappa, or NaN if either series has fewer than 2 unique labels."""
    try:
        if len(set(a)) < 2 or len(set(b)) < 2:
            return float("nan")
        return cohen_kappa_score(a, b)
    except ValueError:
        return float("nan")

overall_kappa = safe_kappa(labels_df["label_a"], labels_df["label_b"])
kappa_rows = [{"scope": "overall", "kappa": round(overall_kappa, 4), "n": len(labels_df)}]
for domain in labels_df["domain"].unique():
    mask = labels_df["domain"] == domain
    k = safe_kappa(labels_df.loc[mask, "label_a"], labels_df.loc[mask, "label_b"])
    kappa_rows.append({
        "scope": domain,
        "kappa": round(k, 4) if not pd.isna(k) else float("nan"),
        "n": int(mask.sum())
    })

kappa_df = pd.DataFrame(kappa_rows)
print("-- Cohen's Kappa (Labeler Agreement) --")
print(kappa_df.to_markdown(index=False))

print("\n-- Label Distribution (label_final = consensus/adjudicated) --")
dist = labels_df.groupby(["domain", "label_final"]).size().unstack(fill_value=0)
dist = dist.reindex(columns=NPS_CLASSES, fill_value=0)
print(dist.to_markdown())

print("\n-- Label Distribution (label_consensus) --")
dist2 = labels_df.groupby(["domain", "label_consensus"]).size().unstack(fill_value=0)
dist2 = dist2.reindex(columns=NPS_CLASSES, fill_value=0)
print(dist2.to_markdown())

print(f"\nOverall agreement rate: {labels_df['agree'].mean():.2%}")
print(f"Labeler A: detractor<={A_DETRACTOR_THRESHOLD}, promoter>={A_PROMOTER_THRESHOLD}, cap={A_SCALE_CAP}")
print(f"Labeler B passive zone: [{B_DETRACTOR_THRESHOLD}, {B_PROMOTER_THRESHOLD})")
_disagree_mask = labels_df["agree"] == 0
_src_counts = labels_df.loc[_disagree_mask, "consensus_source"].value_counts()
print(f"Disagreement resolution (n={_disagree_mask.sum()}): {_src_counts.to_dict()}")

labels_path = f"{CONFIG['outputs_dir']}/labels/labels_df.csv"
labels_df.to_csv(labels_path, index=False)
kappa_df.to_csv(f"{CONFIG['outputs_dir']}/labels/kappa_scores.csv", index=False)
print(f"\nLabels saved ({len(labels_df)} rows). Kappa scores saved.")


Loading sentiment model 'distilbert-base-uncased-finetuned-sst-2-english' (GPU) ...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Running batched sentiment inference: 32,525 texts | batch_size=64 | tail=1500 chars ...
Sentiment done. mean=-0.2951  std=0.8878  min=-0.9998  max=0.9998
Sentiment model released from memory.
-- Cohen's Kappa (Labeler Agreement) --
| scope                                       |   kappa |     n |
|:--------------------------------------------|--------:|------:|
| overall                                     |  0.0204 | 32525 |
| medicare_inbound                            |  0.0206 | 29994 |
| medical_equipment_outbound                  |  0.005  |   738 |
| automotive_and_healthcare_insurance_inbound |  0.0461 |  1793 |

-- Label Distribution (label_final = consensus/adjudicated) --
| domain                                      |   detractor |   passive |   promoter |
|:--------------------------------------------|------------:|----------:|-----------:|
| automotive_and_healthcare_insurance_inbound |        1158 |        66 |        569 |
| medical_equipment_outbound                  |

## § Golden Dataset Validation

Load a human-reviewed Excel file as a **golden reference dataset**.  
This provides ground-truth labels for a subset of calls, enabling:
- Validation of synthetic labelers (Cohen's κ: Human ↔ A, Human ↔ B)
- Override of synthetic labels with verified human labels (higher confidence)
- Rigorous academic evaluation of the labeling methodology

> **Note**: The golden dataset Excel should contain the configured `call_id_col` and `label_col` columns.  
> Set `CONFIG['golden_dataset']['enabled'] = False` to skip.

In [8]:
# -- Golden Dataset: Human-Reviewed Labels ---------------------------------

golden_cfg = CONFIG["golden_dataset"]
golden_df = None
golden_kappa_df = None

def _norm_call_id(x):
    key = normalize_text(str(x)).lower()
    if key.endswith(".json"):
        key = key[:-5]
    return key

def _call_id_key_variants(value, domain=None):
    """Create safe match keys for raw ids, domain::ids, and domain__filename ids."""
    base = _norm_call_id(value)
    keys = {base}
    if "::" in base:
        keys.add(base.split("::", 1)[1])
    domains = []
    if domain is not None and str(domain).strip():
        domains.append(_norm_call_id(domain))
    domains.extend([_norm_call_id(d) for d in CONFIG["healthcare_domains"].keys()])
    for key in list(keys):
        for d in domains:
            prefix = f"{d}__"
            if key.startswith(prefix):
                keys.add(key[len(prefix):])
    expanded = set(keys)
    for key in list(keys):
        if key.endswith("_transcript"):
            expanded.add(key[:-len("_transcript")])
    return sorted(k for k in expanded if k and k != "nan")

# Verify normalisation edge cases -- pure whitespace and "nan" must
# produce zero keys to prevent false-positive golden label matches.
assert _call_id_key_variants("   ", None) == [], "Whitespace IDs must produce no keys"
assert _call_id_key_variants("nan", None) == [], "'nan' IDs must produce no keys"

def _explode_match_keys(df, id_cols):
    rows = []
    for row_idx, row in df.reset_index(drop=True).iterrows():
        for col in id_cols:
            if col not in row or pd.isna(row[col]):
                continue
            for key in _call_id_key_variants(row[col], row.get("domain")):
                rows.append({"_row_idx": row_idx, "_match_key": key})
    if not rows:
        return df.assign(_match_key=pd.Series(dtype=str)).iloc[0:0]
    key_df = pd.DataFrame(rows).drop_duplicates(subset=["_row_idx", "_match_key"])
    base_df = df.reset_index(drop=True).reset_index().rename(columns={"index": "_row_idx"})
    return base_df.merge(key_df, on="_row_idx").drop(columns="_row_idx")

def _resolve_golden_path(configured_path):
    configured = Path(configured_path).expanduser()
    checked = []

    def add_candidate(path):
        if path is None:
            return
        path = Path(path)
        if path not in checked:
            checked.append(path)

    if configured.is_absolute():
        add_candidate(configured)
    else:
        add_candidate(Path.cwd() / configured)
        add_candidate(Path(CONFIG["project_root"]) / configured)
        add_candidate(Path(CONFIG["data_dir"]) / configured)
        add_candidate(Path("/content") / configured)

    for candidate in checked:
        if candidate.exists():
            return candidate, checked

    # Last-resort exact filename search in common Colab locations. Kept shallow-ish
    # by searching only for the configured filename, not arbitrary Excel files.
    if not configured.is_absolute():
        for root in [Path.cwd(), Path("/content")]:
            if root.exists():
                try:
                    for match in root.rglob(configured.name):
                        add_candidate(match)
                        if match.exists():
                            return match, checked
                except Exception:
                    pass

    return checked[0] if checked else configured, checked

if not golden_cfg["enabled"]:
    print("Warning: Golden dataset integration disabled in CONFIG. Skipping.")
else:
    golden_path, golden_checked_paths = _resolve_golden_path(golden_cfg["file_path"])
    call_id_col = str(golden_cfg["call_id_col"]).strip()
    label_col = str(golden_cfg["label_col"]).strip()
    valid_labels = golden_cfg["valid_labels"]

    try:
        golden_raw = pd.read_excel(golden_path, engine="openpyxl")
        golden_raw.columns = golden_raw.columns.astype(str).str.strip()
        print(f"Golden dataset loaded: {len(golden_raw)} rows from {golden_path}")
    except FileNotFoundError:
        print(f"Warning: Golden dataset file not found: {golden_path}")
        print("  Checked these paths:")
        for p in golden_checked_paths:
            print(f"    - {p}")
        print("  Upload nps_classified_calls.xlsx to /content or update CONFIG['golden_dataset']['file_path'].")
        print("  Human-label evaluations will be skipped; synthetic-label metrics are not true NPS performance.")
        golden_cfg["enabled"] = False
        golden_raw = None
    except Exception as e:
        print(f"Warning: Error loading golden dataset: {e}")
        golden_cfg["enabled"] = False
        golden_raw = None

if golden_cfg["enabled"] and golden_raw is not None:
    missing_cols = [c for c in [call_id_col, label_col] if c not in golden_raw.columns]
    if missing_cols:
        print(f"Warning: Golden dataset missing required columns: {missing_cols}")
        print(f"  Available columns: {list(golden_raw.columns)}")
        golden_cfg["enabled"] = False
    else:
        golden_cols = [call_id_col, label_col] + (["domain"] if "domain" in golden_raw.columns else [])
        golden_clean = golden_raw[golden_cols].copy()
        golden_clean[call_id_col] = golden_clean[call_id_col].astype(str).str.strip()
        golden_clean[label_col] = golden_clean[label_col].astype(str).str.strip().str.lower()

        invalid_mask = ~golden_clean[label_col].isin(valid_labels)
        if invalid_mask.any():
            print(f"Warning: Dropping {invalid_mask.sum()} rows with invalid labels: "
                  f"{golden_clean.loc[invalid_mask, label_col].unique()}")
            golden_clean = golden_clean[~invalid_mask].copy()

        golden_keyed = _explode_match_keys(golden_clean, [call_id_col])
        conflict_counts = golden_keyed.groupby("_match_key")[label_col].nunique()
        conflicting_keys = set(conflict_counts[conflict_counts > 1].index)
        if conflicting_keys:
            examples = golden_keyed[golden_keyed["_match_key"].isin(conflicting_keys)]\
                .sort_values("_match_key")\
                .head(10)
            raise ValueError(
                "Conflicting labels found for duplicate golden call_id values. "
                f"Examples:\n{examples[[call_id_col, label_col]].to_string(index=False)}"
            )
        golden_keyed = golden_keyed.drop_duplicates(subset=["_match_key"]).copy()

        label_keys = _explode_match_keys(labels_df, ["call_id", "source_call_id"])
        key_counts = label_keys.groupby("_match_key")["call_id"].nunique()
        ambiguous_keys = set(key_counts[key_counts > 1].index)
        if ambiguous_keys:
            print(f"Warning: Excluding {len(ambiguous_keys)} ambiguous golden match keys shared by multiple calls.")
            label_keys = label_keys[~label_keys["_match_key"].isin(ambiguous_keys)].copy()
        label_keys = label_keys.drop_duplicates(subset=["_match_key", "call_id"])

        golden_df = label_keys.merge(
            golden_keyed[["_match_key", label_col]].rename(columns={label_col: "human_label"}),
            on="_match_key",
            how="inner",
        ).drop_duplicates(subset=["call_id"]).drop(columns=["_match_key"])

        print(f"Matched {len(golden_df)} calls between golden dataset and labels_df")
        print(f"  (out of {golden_clean[call_id_col].nunique()} valid golden rows and {len(labels_df)} labeled rows)")

        if len(golden_df) == 0:
            print("Warning: No matching call_ids found after normalization/domain-prefix stripping.")
            print(f"  Example golden keys: {golden_keyed['_match_key'].head(5).tolist()}")
            print(f"  Example label keys: {label_keys['_match_key'].head(5).tolist()}")
            golden_cfg["enabled"] = False
        else:
            print("\n-- Human Label Distribution (Golden Dataset) --")
            print(golden_df["human_label"].value_counts().reindex(valid_labels, fill_value=0).to_markdown())

            golden_kappa_df = pd.DataFrame([
                {"pair": "Human vs Labeler A", "kappa": round(safe_kappa(golden_df["human_label"], golden_df["label_a"]), 4)},
                {"pair": "Human vs Labeler B", "kappa": round(safe_kappa(golden_df["human_label"], golden_df["label_b"]), 4)},
                {"pair": "Human vs label_final", "kappa": round(safe_kappa(golden_df["human_label"], golden_df["label_final"]), 4)},
                {"pair": "Labeler A vs B (golden subset)", "kappa": round(safe_kappa(golden_df["label_a"], golden_df["label_b"]), 4)},
            ])
            print("\n-- Cohen's Kappa (Golden Dataset Validation) --")
            print(golden_kappa_df.to_markdown(index=False))

            print("\n-- Agreement Rates --")
            print(f"  Human vs Labeler A:   {(golden_df['human_label'] == golden_df['label_a']).mean():.2%}")
            print(f"  Human vs Labeler B:   {(golden_df['human_label'] == golden_df['label_b']).mean():.2%}")
            print(f"  Human vs label_final: {(golden_df['human_label'] == golden_df['label_final']).mean():.2%}")

            cm = confusion_matrix(golden_df["human_label"], golden_df["label_final"], labels=valid_labels)
            cm_df = pd.DataFrame(
                cm,
                index=[f"human:{x[:3]}" for x in valid_labels],
                columns=[f"synth:{x[:3]}" for x in valid_labels],
            )
            print("\n-- Confusion Matrix (Human vs Synthetic label_final) --")
            print(cm_df.to_markdown())

            if golden_cfg["override_synthetic"]:
                golden_map = dict(zip(golden_df["call_id"], golden_df["human_label"]))
                mask = labels_df["call_id"].isin(golden_map)
                labels_df.loc[mask, "label_final"] = labels_df.loc[mask, "call_id"].map(golden_map)
                labels_df.loc[mask, "label_consensus"] = labels_df.loc[mask, "call_id"].map(golden_map)
                labels_df.loc[mask, "confidence"] = golden_cfg["confidence_boost"]
                golden_df.loc[:, "label_final"] = golden_df["human_label"]
                golden_df.loc[:, "label_consensus"] = golden_df["human_label"]
                labels_path = f"{CONFIG['outputs_dir']}/labels/labels_df.csv"
                labels_df.to_csv(labels_path, index=False)
                print(f"\nOverridden {mask.sum()} labels with golden human labels and re-saved labels_df.")

            golden_dir = f"{CONFIG['outputs_dir']}/golden"
            os.makedirs(golden_dir, exist_ok=True)
            golden_df.to_csv(f"{golden_dir}/golden_merged.csv", index=False)
            golden_kappa_df.to_csv(f"{golden_dir}/golden_kappa.csv", index=False)
            cm_df.to_csv(f"{golden_dir}/golden_confusion_matrix.csv")
            print(f"\nGolden dataset analysis saved to {golden_dir}/")

Golden dataset loaded: 429 rows from /content/nps_classified_calls.xlsx
Matched 246 calls between golden dataset and labels_df
  (out of 429 valid golden rows and 32525 labeled rows)

-- Human Label Distribution (Golden Dataset) --
| human_label   |   count |
|:--------------|--------:|
| detractor     |      20 |
| passive       |     215 |
| promoter      |      11 |

-- Cohen's Kappa (Golden Dataset Validation) --
| pair                           |   kappa |
|:-------------------------------|--------:|
| Human vs Labeler A             | -0.0099 |
| Human vs Labeler B             |  0.009  |
| Human vs label_final           |  0.009  |
| Labeler A vs B (golden subset) |  0.0303 |

-- Agreement Rates --
  Human vs Labeler A:   28.46%
  Human vs Labeler B:   9.76%
  Human vs label_final: 9.76%

-- Confusion Matrix (Human vs Synthetic label_final) --
|           |   synth:det |   synth:pas |   synth:pro |
|:----------|------------:|------------:|------------:|
| human:det |          13 

## § Modeling

Train all enabled models with run-mode dispatch (ablation control).  
Linear models use a `ColumnTransformer` to scale only Track B features,  
preserving the TF-IDF/SVD feature space. LightGBM uses raw features.

| Model | Preprocessing | Key Parameters |
|---|---|---|
| LogisticRegression | ColumnTransformer (scale Track B) | max_iter=3000, balanced |
| LinearSVC | ColumnTransformer (scale Track B) | class_weight=balanced; calibration disabled by default |
| LightGBM | None (tree-based) | 800 trees, lr=0.05, class_weight=balanced |

In [9]:
# -- Modeling: Pipeline-Integrated TF-IDF/SVD + Train All Enabled Models --

SEED       = CONFIG["random_seed"]
run_mode   = CONFIG["run_mode"]
target_col = CONFIG.get("training_label_col", "label_final")

# TextToSVD: Per-fold TF-IDF + SVD (prevents data leakage)
class TextToSVD(BaseEstimator, TransformerMixin):
    """TF-IDF -> TruncatedSVD transformer that re-fits per fold to prevent data leakage."""
    def __init__(self, max_features=50000, ngram_range=(1, 2), min_df=3,
                 max_df=0.95, sublinear_tf=True, n_components=200, random_state=42):
        self.max_features  = max_features
        self.ngram_range   = ngram_range
        self.min_df        = min_df
        self.max_df        = max_df
        self.sublinear_tf  = sublinear_tf
        self.n_components  = n_components
        self.random_state  = random_state

    def fit(self, X, y=None):
        texts = X.values.ravel() if hasattr(X, "values") else np.asarray(X).ravel()
        self.tfidf_ = TfidfVectorizer(
            max_features=self.max_features, ngram_range=self.ngram_range,
            min_df=self.min_df, max_df=self.max_df, sublinear_tf=self.sublinear_tf
        )
        tfidf_mat = self.tfidf_.fit_transform(texts)
        n_comp = min(self.n_components, tfidf_mat.shape[1])
        self.svd_ = TruncatedSVD(n_components=n_comp, random_state=self.random_state)
        self.svd_.fit(tfidf_mat)
        self.n_features_in_ = 1
        return self

    def transform(self, X):
        texts = X.values.ravel() if hasattr(X, "values") else np.asarray(X).ravel()
        tfidf_mat = self.tfidf_.transform(texts)
        return self.svd_.transform(tfidf_mat)

    # get_feature_names_out() for sklearn >=1.0 introspection (SHAP, etc.)
    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, ["svd_"])
        return np.array([f"svd_{i:03d}" for i in range(self.svd_.n_components)])


# Fix #1 — Circular feature fix: CueStrippedTextToSVD
# ─────────────────────────────────────────────────────────────────────────
# TF-IDF over raw transcripts learns the labelers' own keyword vocabulary
# (escalation, resolution, thanks, …) as classification signal — not because
# those words predict NPS, but because they ARE the label rule.  This makes
# all synthetic-label CV/holdout metrics circular: they measure how well the
# model reproduces the heuristic, not how well it predicts real NPS.
#
# CueStrippedTextToSVD strips those terms before the vectoriser is fitted so
# the model must find *other* linguistic signals.  It is used exclusively in
# the golden_human_decontam_cv evaluation (human labels, cue-free features)
# which is the only honest accuracy number.
class CueStrippedTextToSVD(TextToSVD):
    """TextToSVD with labeler cue-word masking applied before TF-IDF.

    Strips the keyword vocabulary that both labelers use to assign labels
    (escalation, resolution, thanks, apology, etc.) so the vectoriser
    cannot trivially recover the label from cue-word frequencies.  This
    forces the model to learn from residual linguistic signals and gives a
    decontaminated performance estimate against human-verified labels.
    """
    # Mirrors the patterns in PATTERNS / _POS_RE / _NEG_RE used by both labelers.
    _CUE_PAT = re.compile(
        r"(?i)\b(?:"
        r"supervisor|manager|escalat(?:e|ion|ing)|complain(?:t|ing)|cancel|refund|lawsuit|"
        r"scheduled|appointment|confirm(?:ed|ation)|submitted|resolved|no\s+problem|"
        r"cannot|can't|cant|unable|doesn't\s+work|doesnt\s+work|not\s+coming\s+up|"
        r"wrong|issue|problem|"
        r"i'm\s+sorry|apologize|our\s+apologies|sorry\s+about|"
        r"thanks?|thank\s+you|appreciate|appreciated|"
        r"great|awesome|perfect|excellent|helpful|amazing|good|love|works|"
        r"happy|satisfied|easy|clear|angry|mad|upset|terrible|horrible|"
        r"ridiculous|frustrated|frustrating|broken|delay(?:ed)?|denied|"
        r"difficult|confusing"
        r")\b"
    )

    def _strip(self, texts):
        return np.array([self._CUE_PAT.sub(" ", str(t)) for t in texts])

    def fit(self, X, y=None):
        raw = X.values.ravel() if hasattr(X, "values") else np.asarray(X).ravel()
        return super().fit(pd.Series(self._strip(raw)), y)

    def transform(self, X):
        raw = X.values.ravel() if hasattr(X, "values") else np.asarray(X).ravel()
        return super().transform(pd.Series(self._strip(raw)))


# Fix #5 — Near-duplicate coarse signature
# ─────────────────────────────────────────────────────────────────────────
# Exact-SHA1 grouping misses near-duplicates (minor ASR-noise variants of
# the same call).  A coarse hash over the first and last N chars of the
# normalised transcript groups near-duplicates into the same CV bucket.
def _near_dup_key(text: str,
                  prefix_len: int = 300,
                  suffix_len: int = 300) -> str:
    """Return a coarse near-duplicate signature from text head + tail."""
    norm = re.sub(r"\s+", " ", (text or "").strip()).lower()
    head = norm[:prefix_len]
    tail = norm[-suffix_len:] if len(norm) > suffix_len else norm
    return hashlib.sha1((head + "|||" + tail).encode("utf-8", errors="ignore")).hexdigest()


# Assemble modeling DataFrame
model_df = features_df.merge(
    labels_df[["call_id", "domain", "direction", "text_norm_sha1",
               "label_final", "confidence", "label_consensus"]],
    on="call_id", how="inner"
)
# Fix #5: compute near-duplicate coarse signature for leakage-safe CV grouping.
_prefix = int(CONFIG.get("near_dup_prefix_len", 300))
_suffix = int(CONFIG.get("near_dup_suffix_len", 300))
model_df["near_dup_sha1"] = model_df["text_norm"].apply(
    lambda t: _near_dup_key(t, _prefix, _suffix)
)
n_near_dup_groups = model_df["near_dup_sha1"].nunique()
n_exact_groups    = model_df["text_norm_sha1"].nunique()
print(f"Near-dup groups: {n_near_dup_groups}  (exact SHA1 groups: {n_exact_groups}; "
      f"collapsed {n_exact_groups - n_near_dup_groups} near-dup clusters)")
print(f"Modeling DataFrame: {model_df.shape}")
if target_col not in model_df.columns:
    raise ValueError(f"Configured training_label_col not found in model_df: {target_col}")
print(f"Synthetic training/evaluation target: {target_col}")

# Select feature columns based on run mode
ID_COLS = {"call_id", "source_call_id", "text_norm"}
track_b_cols = [c for c in struct_df.columns if c not in ID_COLS and c not in LABELER_CUE_COLS]

if run_mode == "text_only":
    feature_cols = ["text_norm"]
    use_text = True
    active_b = []
elif run_mode == "structured_only":
    feature_cols = list(track_b_cols)
    use_text = False
    active_b = list(track_b_cols)
else:  # text_plus_structured
    feature_cols = ["text_norm"] + list(track_b_cols)
    use_text = True
    active_b = list(track_b_cols)

# Assert that no labeler cue columns leaked into feature_cols.
_cue_overlap = set(feature_cols) & set(LABELER_CUE_COLS)
assert not _cue_overlap, (
    f"LABELER_CUE_COLS leaked into model feature_cols: {_cue_overlap}. "
    "This causes circular features and inflates synthetic-label metrics."
)
print(f" OK: No labeler cue columns in feature_cols ({len(feature_cols)} features)")

print(f"Run mode: {run_mode}")
print(f"  Text features (TF-IDF->SVD per fold): {'Yes (' + str(CONFIG['svd_components']) + ' components)' if use_text else 'No'}")
print(f"  Track B features: {len(active_b)}")
print(f"  Feature columns: {len(feature_cols)}")

text_svd = TextToSVD(
    max_features=CONFIG["tfidf_max_features"],
    ngram_range=CONFIG["tfidf_ngram_range"],
    min_df=CONFIG["tfidf_min_df"],
    max_df=CONFIG["tfidf_max_df"],
    n_components=CONFIG["svd_components"],
    random_state=SEED,
)

# Use registry-based scaler (validated at config load time).
scaler_cls = SCALER_REGISTRY[CONFIG["track_b_scaler"]]

# Use deepcopy (not clone) so each pipeline owns its own TextToSVD
# instance with no shared fitted-state references across folds or model types.
linear_transformers = []
if use_text:
    linear_transformers.append(("text_svd", deepcopy(text_svd), ["text_norm"]))
if active_b:
    linear_transformers.append(("track_b", scaler_cls(), active_b))
linear_preprocessor = ColumnTransformer(linear_transformers, remainder="drop") if linear_transformers else "passthrough"

tree_transformers = []
if use_text:
    tree_transformers.append(("text_svd", deepcopy(text_svd), ["text_norm"]))
if active_b:
    tree_transformers.append(("track_b", "passthrough", active_b))
tree_preprocessor = ColumnTransformer(tree_transformers, remainder="drop") if tree_transformers else "passthrough"

def build_model_registry(linear_preprocessor_template, tree_preprocessor_template):
    """Create enabled model pipelines from CONFIG for any feature/preprocessor set."""
    models = {}

    if CONFIG["models"]["logreg"]["enabled"]:
        logreg_cfg = CONFIG["models"]["logreg"]
        models["logreg"] = Pipeline([
            ("preprocessor", deepcopy(linear_preprocessor_template)),
            ("clf", LogisticRegression(
                max_iter=logreg_cfg["max_iter"],
                class_weight=logreg_cfg["class_weight"],
                random_state=SEED, n_jobs=-1
            ))
        ])

    if CONFIG["models"]["linearsvc"]["enabled"]:
        svc_cfg = CONFIG["models"]["linearsvc"]
        svc_clf = LinearSVC(class_weight=svc_cfg["class_weight"], random_state=SEED, max_iter=5000)
        if svc_cfg.get("calibration", True):
            svc_clf = CalibratedClassifierCV(
                svc_clf,
                method=svc_cfg.get("calibration_method", "sigmoid"),
                cv=svc_cfg.get("calibration_cv", 3),
            )
        models["linearsvc"] = Pipeline([
            ("preprocessor", deepcopy(linear_preprocessor_template)),
            ("clf", svc_clf),
        ])

    if CONFIG["models"]["lgbm"]["enabled"]:
        lgbm_cfg = CONFIG["models"]["lgbm"]
        models["lgbm"] = Pipeline([
            ("preprocessor", deepcopy(tree_preprocessor_template)),
            ("clf", LGBMClassifier(
                n_estimators=lgbm_cfg["n_estimators"],
                learning_rate=lgbm_cfg["learning_rate"],
                num_leaves=lgbm_cfg["num_leaves"],
                class_weight=lgbm_cfg.get("class_weight"),
                subsample=lgbm_cfg["subsample"],
                colsample_bytree=lgbm_cfg["colsample_bytree"],
                random_state=SEED, verbosity=-1,
            ))
        ])

    return models

trained_models = build_model_registry(linear_preprocessor, tree_preprocessor)

def _get_calibrator(model):
    if isinstance(model, CalibratedClassifierCV):
        return model
    if isinstance(model, Pipeline) and isinstance(model.named_steps.get("clf"), CalibratedClassifierCV):
        return model.named_steps["clf"]
    return None

def _requested_calibration_cv(calibrator):
    cv = getattr(calibrator, "cv", None)
    if cv is None: return 5
    if isinstance(cv, (int, np.integer)): return int(cv)
    n_splits = getattr(cv, "n_splits", None)
    return int(n_splits) if isinstance(n_splits, (int, np.integer)) else None

def _disable_pipeline_calibration(model, calibrator):
    if not isinstance(model, Pipeline): return False
    base = getattr(calibrator, "estimator", None) or getattr(calibrator, "base_estimator", None)
    if base is None: return False
    model.set_params(clf=clone(base))
    return True

def fit_model(model, X, y, w, model_name):
    """Fit model with sample weights; warn and fall back gracefully if unsupported.

    Graceful fallback with UserWarning instead of raising TypeError,
    making the pipeline robust to estimators that lack sample_weight support.
    """
    y_counts = pd.Series(y).dropna().value_counts()
    if len(y_counts) < 2:
        raise ValueError(
            f"Cannot fit {model_name}: training labels have fewer than two classes: {y_counts.to_dict()}"
        )

    calibrator = _get_calibrator(model)
    if model_name == "linearsvc" and calibrator is not None:
        requested_cv  = _requested_calibration_cv(calibrator)
        min_class_cnt = int(y_counts.min())
        if requested_cv is not None and min_class_cnt < 2:
            if _disable_pipeline_calibration(model, calibrator):
                calibrator = None
                print(f"  Disabled LinearSVC calibration; smallest class has fewer than 2 rows: {y_counts.to_dict()}")
            else:
                raise ValueError(
                    f"Cannot calibrate {model_name}: each class needs >=2 rows, got {y_counts.to_dict()}"
                )
        elif requested_cv is not None and min_class_cnt < requested_cv:
            calibrator.cv = max(2, min_class_cnt)
            print(f"  Adjusted LinearSVC calibration cv to {calibrator.cv}: {y_counts.to_dict()}")

    if w is None:
        model.fit(X, y)
        return

    # Try each weight-passing strategy; warn and fall back if all fail.
    if isinstance(model, Pipeline):
        try:
            model.fit(X, y, clf__sample_weight=w)
            return
        except TypeError:
            pass

    if isinstance(model, CalibratedClassifierCV):
        try:
            model.fit(X, y, sample_weight=w)
            return
        except TypeError:
            pass

    try:
        model.fit(X, y, sample_weight=w)
        return
    except TypeError:
        warnings.warn(
            f"{model_name} does not support sample_weight; fitting without weights.",
            UserWarning, stacklevel=2,
        )
        model.fit(X, y)

# Train on full dataset (for saving)
X_full = model_df[feature_cols]
y_full = model_df[target_col]
w_full = model_df["confidence"]

class_counts = y_full.value_counts().reindex(CONFIG["golden_dataset"]["valid_labels"], fill_value=0)
print("\nModel target distribution:")
print(class_counts.to_markdown())
if (class_counts > 0).sum() < 2:
    raise ValueError(f"Cannot train classifiers: {target_col} has fewer than two classes: {class_counts.to_dict()}")

for name, model in trained_models.items():
    print(f"\nTraining {name} on full dataset...")
    fit_model(model, X_full, y_full, w_full, name)
    model_path = f"{CONFIG['outputs_dir']}/models/{name}_model.joblib"
    joblib.dump(model, model_path)
    print(f"  v {name} trained and saved.")

print(f"\nv All {len(trained_models)} models trained and saved.")


Modeling DataFrame: (32525, 28)
Synthetic training/evaluation target: label_consensus
 OK: No labeler cue columns in feature_cols (15 features)
Run mode: text_plus_structured
  Text features (TF-IDF->SVD per fold): Yes (200 components)
  Track B features: 14
  Feature columns: 15

Model target distribution:
| label_consensus   |   count |
|:------------------|--------:|
| detractor         |   20995 |
| passive           |     869 |
| promoter          |   10661 |

Training logreg on full dataset...
  v logreg trained and saved.

Training linearsvc on full dataset...
  v linearsvc trained and saved.

Training lgbm on full dataset...
  v lgbm trained and saved.

v All 3 models trained and saved.


## § Evaluation

### Domain-Stratified Holdout (Primary)
3 rotations: train on 2 domains, test on the held-out domain.

### Stratified Grouped 5-Fold CV (Secondary)
Pool all data with group-aware splitting (prevents duplicate transcript leakage).

### Metrics
- **Primary**: macro-F1
- **Secondary**: balanced accuracy, accuracy, weighted-F1, per-class precision/recall
- **Calibration**: Brier score (one-vs-rest, averaged) for probabilistic models

In [10]:
# ── Evaluation: Domain Holdout + Grouped CV + Probability Metrics ────────
# Fixed-class metrics prevent single-class test splits from inflating macro-F1.
# Domain holdout removes exact duplicate transcript hashes from training.

metrics_records = []
metrics_path = f"{CONFIG['outputs_dir']}/metrics/metrics_v1.csv"
significance_path = f"{CONFIG['outputs_dir']}/metrics/model_significance_tests.csv"
METRIC_COLUMNS = ["run_type", "label_source", "test_domain", "model", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "n_train", "n_test"]
metrics_df = pd.DataFrame(columns=METRIC_COLUMNS)
CLASS_ORDER = ["detractor", "passive", "promoter"]

def _save_metrics_checkpoint(stage, records=None):
    """Persist metrics checkpoin Pure function; no global mutation.
    Caller assigns the return value. Passing records= avoids silent overwrites.
    """
    if records is None:
        records = metrics_records
    df = pd.DataFrame(records) if records else pd.DataFrame(columns=METRIC_COLUMNS)
    df.to_csv(metrics_path, index=False)
    print(f"  Metrics checkpoint saved after {stage}: {len(df)} rows")
    return df

def _as_label_series(y):
    return pd.Series(y).astype(str).reset_index(drop=True)

def _balanced_accuracy_fixed(y_true, y_pred):
    """ Exclude zero-support classes from the macro average.
    Including them depresses balanced_accuracy on splits where a class is absent.
    """
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)
    support = cm.sum(axis=1)
    valid = support > 0
    if not valid.any():
        return 0.0
    recalls = np.diag(cm)[valid] / support[valid]
    return float(np.mean(recalls))

def _get_model_classes(model):
    """Extract class order from a fitted model/pipeline."""
    if hasattr(model, "classes_"):
        return list(model.classes_)
    if hasattr(model, "steps"):
        last = model.steps[-1][1]
        if hasattr(last, "classes_"):
            return list(last.classes_)
    return None

def compute_metrics(y_true, y_pred, y_proba=None, model=None):
    """Compute metrics against the fixed 3-class task definition."""
    y_true = _as_label_series(y_true)
    y_pred = _as_label_series(y_pred)

    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": _balanced_accuracy_fixed(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=CLASS_ORDER, average="weighted", zero_division=0),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=CLASS_ORDER,
        target_names=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
    for cls in CLASS_ORDER:
        m[f"support_{cls}"] = report[cls]["support"]
        m[f"precision_{cls}"] = report[cls]["precision"]
        m[f"recall_{cls}"] = report[cls]["recall"]
        m[f"f1_{cls}"] = report[cls]["f1-score"]

    if y_proba is not None and CONFIG["probability_metrics"]:
        brier_scores = []
        model_classes = _get_model_classes(model) if model is not None else None
        if model_classes is None:
            model_classes = list(CLASS_ORDER)
        for cls in CLASS_ORDER:
            if cls in model_classes:
                col_idx = model_classes.index(cls)
                if col_idx < y_proba.shape[1]:
                    y_bin = (y_true == cls).astype(int)
                    brier_scores.append(brier_score_loss(y_bin, y_proba[:, col_idx]))
        m["brier_score_avg"] = np.mean(brier_scores) if brier_scores else None

    return m

def get_proba(model, X, name):
    """Get probability predictions if available."""
    try:
        return model.predict_proba(X)
    except AttributeError:
        return None

def _can_evaluate_split(y_train, split_name):
    counts = pd.Series(y_train).dropna().value_counts()
    n_train = int(counts.sum())
    min_train = int(CONFIG.get("min_train_samples", 2))
    min_per_class = int(CONFIG.get("min_train_samples_per_class", 1))
    if n_train < min_train:
        print(f"  Skipping {split_name}: training split has {n_train} rows, below min_train_samples={min_train}")
        return False
    if len(counts) < 2:
        print(f"  Skipping {split_name}: training split has fewer than two classes: {counts.to_dict()}")
        return False
    if int(counts.min()) < min_per_class:
        print(f"  Skipping {split_name}: smallest class has {int(counts.min())} rows, below min_train_samples_per_class={min_per_class}: {counts.to_dict()}")
        return False
    return True

def _make_hash_safe_splits(X, y, groups, n_splits, split_name):
    """Return leakage-safe CV splits; fallback splits unique transcript hashes, never rows."""
    try:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        split_iter = list(cv.split(X, y, groups=groups))
        cv_type = "StratifiedGroupKFold"
    except Exception as e:
        print(f"  StratifiedGroupKFold failed ({e}); using hash-group-level StratifiedKFold fallback.")
        tmp = pd.DataFrame({
            "row_pos": np.arange(len(y)),
            "group": pd.Series(groups).astype(str).values,
            "label": pd.Series(y).astype(str).values,
        })
        group_labels = tmp.groupby("group")["label"].agg(lambda s: s.value_counts().index[0])
        group_class_counts = group_labels.value_counts()
        n_splits_eff = min(int(n_splits), int(group_class_counts.min())) if len(group_class_counts) else 0
        if len(group_class_counts) < 2 or n_splits_eff < 2:
            raise ValueError(
                f"No leakage-safe CV fallback available for {split_name}; "
                f"group-level class counts are {group_class_counts.to_dict()}"
            )
        unique_groups = group_labels.index.to_numpy()
        group_y = group_labels.to_numpy()
        group_cv = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=SEED)
        split_iter = []
        for group_train_idx, group_test_idx in group_cv.split(unique_groups, group_y):
            train_groups = set(unique_groups[group_train_idx])
            test_groups = set(unique_groups[group_test_idx])
            train_idx = tmp.loc[tmp["group"].isin(train_groups), "row_pos"].to_numpy()
            test_idx = tmp.loc[tmp["group"].isin(test_groups), "row_pos"].to_numpy()
            split_iter.append((train_idx, test_idx))
        cv_type = f"HashGroupStratifiedKFold fallback (n_splits={n_splits_eff})"

    for fold_idx, (train_idx, test_idx) in enumerate(split_iter):
        train_groups = set(pd.Series(groups).iloc[train_idx].astype(str))
        test_groups = set(pd.Series(groups).iloc[test_idx].astype(str))
        overlap = train_groups.intersection(test_groups)
        if overlap:
            raise ValueError(f"Hash leakage detected in {split_name} fold {fold_idx}: {len(overlap)} shared transcript hashes")
    return split_iter, cv_type
# ══════════════════════════════════════════════════════════════════════════
# DOMAIN-STRATIFIED HOLDOUT
# ══════════════════════════════════════════════════════════════════════════
if CONFIG["domain_holdout"]:
    print("Synthetic-label domain-stratified holdout\n")
    print("  Note: this evaluates synthetic labels, not true NPS performance.")
    domains = model_df["domain"].unique()

    for test_domain in domains:
        train_mask = model_df["domain"] != test_domain
        test_mask = model_df["domain"] == test_domain

        train_hashes = set(model_df.loc[train_mask, "text_norm_sha1"])
        test_hashes = set(model_df.loc[test_mask, "text_norm_sha1"])
        overlap_hashes = train_hashes.intersection(test_hashes)
        if overlap_hashes:
            n_before = int(train_mask.sum())
            train_mask = train_mask & ~model_df["text_norm_sha1"].isin(overlap_hashes)
            print(
                f"  Removed {n_before - int(train_mask.sum())} training rows for {test_domain} "
                f"because their transcript hashes also appear in the held-out domain."
            )

        X_train = model_df.loc[train_mask, feature_cols]
        y_train = model_df.loc[train_mask, target_col]
        w_train = model_df.loc[train_mask, "confidence"]
        X_test = model_df.loc[test_mask, feature_cols]
        y_test = model_df.loc[test_mask, target_col]

        if not _can_evaluate_split(y_train, f"domain_holdout:{test_domain}"):
            continue

        for model_name, model_template in trained_models.items():
            model = clone(model_template)
            fit_model(model, X_train, y_train, w_train, model_name)

            y_pred = model.predict(X_test)
            y_proba = get_proba(model, X_test, model_name)

            m = compute_metrics(y_test, y_pred, y_proba, model=model)
            m["run_type"] = "domain_holdout"
            m["label_source"] = "synthetic"
            m["test_domain"] = test_domain
            m["model"] = model_name
            m["n_train"] = train_mask.sum()
            m["n_test"] = test_mask.sum()
            m["duplicate_hashes_removed"] = len(overlap_hashes)
            metrics_records.append(m)

            cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
            cm_df = pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER)
            cm_path = f"{CONFIG['outputs_dir']}/metrics/confusion_matrices_v1/holdout_{test_domain}_{model_name}.csv"
            cm_df.to_csv(cm_path)

            print(f"  {model_name} → test={test_domain}: macro_f1={m['macro_f1']:.4f}, bal_acc={m['balanced_accuracy']:.4f}")

metrics_df = _save_metrics_checkpoint("domain holdout")

# ══════════════════════════════════════════════════════════════════════════
# STRATIFIED GROUPED 5-FOLD CV
# ══════════════════════════════════════════════════════════════════════════
if CONFIG["pooled_cv"]:
    print("\n═══ Stratified Grouped 5-Fold CV ═══\n")

    groups = model_df["text_norm_sha1"]

    split_iter, cv_type = _make_hash_safe_splits(
        X_full, y_full, groups, CONFIG["cv_splits"], "pooled_cv"
    )

    print(f"  Using {cv_type}")

    for fold_idx, (train_idx, test_idx) in enumerate(split_iter):
        X_tr = X_full.iloc[train_idx]
        y_tr = y_full.iloc[train_idx]
        w_tr = w_full.iloc[train_idx]
        X_te = X_full.iloc[test_idx]
        y_te = y_full.iloc[test_idx]

        if not _can_evaluate_split(y_tr, f"pooled_cv:fold_{fold_idx}"):
            continue

        for model_name, model_template in trained_models.items():
            model = clone(model_template)
            fit_model(model, X_tr, y_tr, w_tr, model_name)

            y_pred = model.predict(X_te)
            y_proba = get_proba(model, X_te, model_name)

            m = compute_metrics(y_te, y_pred, y_proba, model=model)
            m["run_type"] = "pooled_cv"
            m["label_source"] = "synthetic"
            m["test_domain"] = f"fold_{fold_idx}"
            m["model"] = model_name
            m["n_train"] = len(train_idx)
            m["n_test"] = len(test_idx)
            metrics_records.append(m)

            cm = confusion_matrix(y_te, y_pred, labels=CLASS_ORDER)
            cm_df = pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER)
            cm_path = f"{CONFIG['outputs_dir']}/metrics/confusion_matrices_v1/cv_fold{fold_idx}_{model_name}.csv"
            cm_df.to_csv(cm_path)

        print(f"  Fold {fold_idx} complete.")

# ── Save metrics ─────────────────────────────────────────────────────────
metrics_df = _save_metrics_checkpoint("final evaluation")

print("\n═══ Domain Holdout Summary ═══")
holdout_metrics = metrics_df[metrics_df["run_type"] == "domain_holdout"] if len(metrics_df) else pd.DataFrame()
if len(holdout_metrics) > 0:
    summary = holdout_metrics.pivot_table(
        index="test_domain", columns="model",
        values=["macro_f1", "balanced_accuracy"], aggfunc="mean"
    )
    print(summary.to_markdown())

print("\n═══ Cross-Validation Summary (mean ± std) ═══")
cv_metrics = metrics_df[metrics_df["run_type"] == "pooled_cv"] if len(metrics_df) else pd.DataFrame()
if len(cv_metrics) > 0:
    cv_summary = cv_metrics.groupby("model").agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        bal_acc_mean=("balanced_accuracy", "mean"),
        bal_acc_std=("balanced_accuracy", "std"),
    ).round(4)
    print(cv_summary.to_markdown())

def paired_model_significance(metrics_df, run_types=("pooled_cv", "domain_holdout", "golden_human_text_cv"), metrics=("macro_f1", "balanced_accuracy")):
    rows = []
    if metrics_df is None or len(metrics_df) == 0:
        return pd.DataFrame(rows)
    for run_type in run_types:
        data = metrics_df[metrics_df["run_type"] == run_type].copy()
        if len(data) == 0:
            continue
        models = sorted(data["model"].dropna().unique())
        for metric in metrics:
            wide = data.pivot_table(index="test_domain", columns="model", values=metric, aggfunc="mean")
            for model_a, model_b in combinations(models, 2):
                if model_a not in wide or model_b not in wide:
                    continue
                paired = wide[[model_a, model_b]].dropna()
                row = {
                    "run_type": run_type,
                    "metric": metric,
                    "model_a": model_a,
                    "model_b": model_b,
                    "n_pairs": len(paired),
                    "mean_diff_a_minus_b": None,
                    "paired_t_p": None,
                    "wilcoxon_p": None,
                }
                if len(paired) >= 2:
                    diffs = paired[model_a] - paired[model_b]
                    row["mean_diff_a_minus_b"] = float(diffs.mean())
                    row["paired_t_p"] = float(ttest_rel(paired[model_a], paired[model_b]).pvalue)
                    try:
                        row["wilcoxon_p"] = float(wilcoxon(paired[model_a], paired[model_b]).pvalue)
                    except ValueError:
                        row["wilcoxon_p"] = None
                rows.append(row)
    return pd.DataFrame(rows)

significance_df = paired_model_significance(metrics_df)
significance_df.to_csv(significance_path, index=False)
if len(significance_df) > 0:
    print("\n═══ Paired Model Significance Tests ═══")
    print(significance_df.round(6).to_markdown(index=False))

print(f"\n✓ Metrics saved to {metrics_path}")
print(f"✓ Significance tests saved to {significance_path}")


# ======================================================================
# GOLDEN-ONLY EVALUATION (against human-verified labels)
# ======================================================================
if golden_cfg["enabled"] and golden_df is not None and len(golden_df) > 0:
    print("\n=== Golden Dataset Evaluation (vs Human Labels) ===\n")

    golden_ids = set(golden_df["call_id"].values)
    golden_mask = model_df["call_id"].isin(golden_ids)

    if golden_mask.sum() > 0:
        train_mask = ~golden_mask
        test_mask = golden_mask
        golden_test_hashes = set(model_df.loc[test_mask, "text_norm_sha1"])
        n_train_before_hash_filter = int(train_mask.sum())
        train_mask = train_mask & ~model_df["text_norm_sha1"].isin(golden_test_hashes)
        n_removed_hash_overlap = n_train_before_hash_filter - int(train_mask.sum())
        if n_removed_hash_overlap:
            print(
                f"  Removed {n_removed_hash_overlap} training rows from golden_eval "
                "because their transcript hashes also appear in the golden test set."
            )

        X_train_g = model_df.loc[train_mask, feature_cols]
        y_train_g = model_df.loc[train_mask, target_col]
        w_train_g = model_df.loc[train_mask, "confidence"]
        X_test_g = model_df.loc[test_mask, feature_cols]

        golden_labels_map = dict(zip(golden_df["call_id"], golden_df["human_label"]))
        y_test_g = model_df.loc[test_mask, "call_id"].map(golden_labels_map)

        if _can_evaluate_split(y_train_g, "golden_eval"):
            for model_name, model_template in trained_models.items():
                model = clone(model_template)
                fit_model(model, X_train_g, y_train_g, w_train_g, model_name)

                y_pred = model.predict(X_test_g)
                y_proba = get_proba(model, X_test_g, model_name)

                m = compute_metrics(y_test_g, y_pred, y_proba, model=model)
                m["run_type"] = "golden_eval"
                m["label_source"] = "human_test_synthetic_train"
                m["test_domain"] = "golden_all"
                m["model"] = model_name
                m["n_train"] = train_mask.sum()
                m["n_test"] = test_mask.sum()
                m["duplicate_hash_rows_removed"] = n_removed_hash_overlap
                metrics_records.append(m)

                cm = confusion_matrix(y_test_g, y_pred, labels=CLASS_ORDER)
                cm_df = pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER)
                cm_path = f"{CONFIG['outputs_dir']}/metrics/confusion_matrices_v1/golden_{model_name}.csv"
                cm_df.to_csv(cm_path)

                print(f"  {model_name} vs HUMAN labels: macro_f1={m['macro_f1']:.4f}, bal_acc={m['balanced_accuracy']:.4f}")

            metrics_df = pd.DataFrame(metrics_records)
            metrics_df.to_csv(metrics_path, index=False)

            print("\n=== Golden Evaluation Summary ===")
            golden_metrics = metrics_df[metrics_df["run_type"] == "golden_eval"]
            if len(golden_metrics) > 0:
                print(golden_metrics[["model", "macro_f1", "balanced_accuracy"]].to_markdown(index=False))
    else:
        print("  No golden samples found in model_df.")

Synthetic-label domain-stratified holdout

  Note: this evaluates synthetic labels, not true NPS performance.
  logreg → test=medicare_inbound: macro_f1=0.3639, bal_acc=0.4264
  linearsvc → test=medicare_inbound: macro_f1=0.4361, bal_acc=0.4385
  lgbm → test=medicare_inbound: macro_f1=0.4332, bal_acc=0.4395
  logreg → test=medical_equipment_outbound: macro_f1=0.2520, bal_acc=0.3721
  linearsvc → test=medical_equipment_outbound: macro_f1=0.4207, bal_acc=0.4247
  lgbm → test=medical_equipment_outbound: macro_f1=0.4013, bal_acc=0.4154
  logreg → test=automotive_and_healthcare_insurance_inbound: macro_f1=0.3148, bal_acc=0.4103
  linearsvc → test=automotive_and_healthcare_insurance_inbound: macro_f1=0.4352, bal_acc=0.4634
  lgbm → test=automotive_and_healthcare_insurance_inbound: macro_f1=0.4481, bal_acc=0.4726
  Metrics checkpoint saved after domain holdout: 9 rows

═══ Stratified Grouped 5-Fold CV ═══

  Using StratifiedGroupKFold


KeyboardInterrupt: 

## Text-Only Ablation Analysis

This section runs two evaluations on the golden (human-reviewed) dataset:

### A — `golden_human_text_cv` (baseline)
Text-only features, human labels.  Structured cue columns are excluded, but
**the raw transcripts still contain the labeler cue words** (escalation, resolution,
thanks, …), so TF-IDF can still learn them.  Performance here reflects how well the
model recovers human labels *when it can see cue words*.

### B — `golden_human_decontam_cv` (**Fix #1 — decontaminated**)
Identical setup, but `CueStrippedTextToSVD` masks the labeler keyword vocabulary
before the vectoriser is fitted.  This is the **only honest accuracy number**: it
measures whether the model learns genuine linguistic patterns beyond the keyword
heuristic, evaluated against human-verified NPS labels.

> The gap between A and B quantifies how much of the reported accuracy is circular
> (i.e., recoverable by simply learning the labeler's own keyword rules).


In [ ]:
# -- Human-Label Text-Only Evaluation on Golden Dataset --------------------
# Both train and test labels are human-reviewed; the model uses only text.

if golden_cfg["enabled"] and golden_df is not None and len(golden_df) > 0:
    print("=== Golden Human-Label Text-Only Cross-Validation ===\n")

    golden_labels_map = dict(zip(golden_df["call_id"], golden_df["human_label"]))
    golden_model_df = model_df[model_df["call_id"].isin(golden_labels_map)].copy()
    golden_model_df["human_label"] = golden_model_df["call_id"].map(golden_labels_map)
    golden_model_df = golden_model_df.dropna(subset=["human_label"])

    human_counts = golden_model_df["human_label"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    print("Human-label class counts used for golden CV:")
    print(human_counts.to_markdown())

    min_class = int(human_counts[human_counts > 0].min()) if (human_counts > 0).any() else 0
    if (human_counts > 0).sum() < 2 or min_class < 2:
        print("  Warning: skipped because golden data lacks enough examples per class.")
    else:
        n_splits = min(CONFIG["cv_splits"], min_class)
        X_g = golden_model_df[["text_norm"]]
        y_g = golden_model_df["human_label"]
        # Fix #5: use near_dup_sha1 so near-duplicates are kept in the same fold.
        groups_g = (
            golden_model_df["near_dup_sha1"]
            if "near_dup_sha1" in golden_model_df.columns
            else golden_model_df["text_norm_sha1"]
        )

        split_iter_g, cv_name_g = _make_hash_safe_splits(
            X_g, y_g, groups_g, n_splits, "golden_human_text_cv"
        )

        print(f"  Using {cv_name_g} with n_splits={n_splits}")
        text_only_preprocessor = ColumnTransformer([
            ("text_svd", clone(text_svd), ["text_norm"]),
        ], remainder="drop")
        text_only_models = build_model_registry(text_only_preprocessor, text_only_preprocessor)

        human_cv_results = []
        for fold_idx, (train_idx, test_idx) in enumerate(split_iter_g):
            X_train_h = X_g.iloc[train_idx]
            y_train_h = y_g.iloc[train_idx]
            X_test_h = X_g.iloc[test_idx]
            y_test_h = y_g.iloc[test_idx]
            w_train_h = pd.Series(np.ones(len(train_idx)), index=X_train_h.index)

            for model_name, model_template in text_only_models.items():
                h_model = clone(model_template)

                fit_model(h_model, X_train_h, y_train_h, w_train_h, model_name)
                y_pred_h = h_model.predict(X_test_h)
                y_proba_h = get_proba(h_model, X_test_h, model_name)

                m = compute_metrics(y_test_h, y_pred_h, y_proba_h, model=h_model)
                m["model"] = model_name
                m["run_type"] = "golden_human_text_cv"
                m["label_source"] = "human_train_human_test"
                m["test_domain"] = f"fold_{fold_idx}"
                m["n_train"] = len(train_idx)
                m["n_test"] = len(test_idx)
                human_cv_results.append(m)

            print(f"  Golden human fold {fold_idx} complete.")

        if human_cv_results:
            human_cv_df = pd.DataFrame(human_cv_results)
            human_cv_path = f"{CONFIG['outputs_dir']}/metrics/golden_human_text_cv.csv"
            human_cv_df.to_csv(human_cv_path, index=False)
            metrics_df = pd.concat([metrics_df, human_cv_df], ignore_index=True)
            metrics_df.to_csv(metrics_path, index=False)
            significance_df = paired_model_significance(metrics_df)
            significance_df.to_csv(significance_path, index=False)
            print("\nGolden human-label CV summary (cue words VISIBLE to TF-IDF):")
            print(human_cv_df.groupby("model")[["macro_f1", "balanced_accuracy"]].mean().round(4).to_markdown())
            print(f"Golden human-label CV saved to {human_cv_path}")

        # ── Fix #1: Decontaminated evaluation ─────────────────────────────────
        # Strip labeler cue words from text before TF-IDF is fitted so the model
        # cannot shortcut by learning keyword heuristics.  This is the only
        # evaluation that measures genuine linguistic generalisation.
        print("\n=== Decontaminated Golden CV (cue words stripped from text — Fix #1) ===")
        print("  Cue words (escalation, resolution, thanks, …) are masked before"
              " TF-IDF is fitted.\n  This breaks the circular labeler→feature loop.")

        decontam_text_svd = CueStrippedTextToSVD(
            max_features=CONFIG["tfidf_max_features"],
            ngram_range=CONFIG["tfidf_ngram_range"],
            min_df=CONFIG["tfidf_min_df"],
            max_df=CONFIG["tfidf_max_df"],
            n_components=CONFIG["svd_components"],
            random_state=SEED,
        )
        decontam_preprocessor = ColumnTransformer([
            ("text_svd", clone(decontam_text_svd), ["text_norm"]),
        ], remainder="drop")
        decontam_models = build_model_registry(decontam_preprocessor, decontam_preprocessor)

        decontam_cv_results = []
        for fold_idx, (train_idx, test_idx) in enumerate(split_iter_g):
            X_train_dc = X_g.iloc[train_idx]
            y_train_dc = y_g.iloc[train_idx]
            X_test_dc  = X_g.iloc[test_idx]
            y_test_dc  = y_g.iloc[test_idx]
            w_train_dc = pd.Series(np.ones(len(train_idx)), index=X_train_dc.index)

            for model_name, model_template in decontam_models.items():
                dc_model = clone(model_template)
                fit_model(dc_model, X_train_dc, y_train_dc, w_train_dc, model_name)
                y_pred_dc  = dc_model.predict(X_test_dc)
                y_proba_dc = get_proba(dc_model, X_test_dc, model_name)

                m = compute_metrics(y_test_dc, y_pred_dc, y_proba_dc, model=dc_model)
                m["model"]        = model_name
                m["run_type"]     = "golden_human_decontam_cv"
                m["label_source"] = "human_train_human_test_cue_stripped"
                m["test_domain"]  = f"fold_{fold_idx}"
                m["n_train"]      = len(train_idx)
                m["n_test"]       = len(test_idx)
                decontam_cv_results.append(m)

            print(f"  Decontaminated fold {fold_idx} complete.")

        if decontam_cv_results:
            decontam_cv_df  = pd.DataFrame(decontam_cv_results)
            decontam_cv_path = f"{CONFIG['outputs_dir']}/metrics/golden_human_decontam_cv.csv"
            decontam_cv_df.to_csv(decontam_cv_path, index=False)
            metrics_df = pd.concat([metrics_df, decontam_cv_df], ignore_index=True)
            metrics_df.to_csv(metrics_path, index=False)
            significance_df = paired_model_significance(metrics_df)
            significance_df.to_csv(significance_path, index=False)

            print("\nDecontaminated CV summary (honest NPS proxy accuracy):")
            print(decontam_cv_df.groupby("model")[["macro_f1", "balanced_accuracy"]].mean().round(4).to_markdown())

            if human_cv_results:
                baseline_mean = human_cv_df.groupby("model")["macro_f1"].mean()
                decontam_mean = decontam_cv_df.groupby("model")["macro_f1"].mean()
                gap = (baseline_mean - decontam_mean).rename("circular_signal_gap")
                print("\nCircular-signal gap (baseline minus decontaminated macro-F1):")
                print("  Positive = some accuracy was from keyword shortcuts, not genuine generalisation.")
                print(gap.round(4).to_markdown())
            print(f"Decontaminated CV saved to {decontam_cv_path}")
else:
    print("Warning: golden human-label text-only CV was not run because the golden dataset was not loaded.")
    print("  Synthetic-label CV/holdout metrics should not be described as true NPS performance.")

## § Figures

Publication-quality visualizations displayed inline and saved as PNGs.  
Style: seaborn whitegrid, dpi=200, tight_layout.

In [ ]:
# ── Publication-Quality Figures ───────────────────────────────────────────

fig_dir = f"{CONFIG['outputs_dir']}/figures"
DPI = 200
metric_cols = ["run_type", "label_source", "test_domain", "model", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "n_train", "n_test"]
if "metrics_df" not in globals():
    metrics_path = f"{CONFIG['outputs_dir']}/metrics/metrics_v1.csv"
    if Path(metrics_path).exists():
        metrics_df = pd.read_csv(metrics_path)
        print(f"Loaded metrics from {metrics_path}")
    else:
        metrics_df = pd.DataFrame(columns=metric_cols)
        print("Warning: metrics_df was not available and no metrics CSV was found; metric figures will be skipped.")
else:
    missing_metric_cols = [c for c in metric_cols if c not in metrics_df.columns]
    for c in missing_metric_cols:
        metrics_df[c] = np.nan

# ── 1. Domain Holdout: Macro-F1 by test-domain × model ──────────────────
if CONFIG["domain_holdout"]:
    holdout = metrics_df[metrics_df["run_type"] == "domain_holdout"].copy()
    if len(holdout) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        pivot = holdout.pivot_table(index="test_domain", columns="model", values="macro_f1", aggfunc="mean")
        pivot.plot(kind="bar", ax=ax, rot=15)
        ax.set_title("Domain Holdout — Macro F1 by Test Domain × Model", fontsize=14)
        ax.set_ylabel("Macro F1")
        ax.set_xlabel("Held-out Domain")
        ax.legend(title="Model")
        ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/domain_holdout_macro_f1.png", dpi=DPI)
        plt.show()
        plt.close()  # release figure memory

        # Balanced Accuracy
        fig, ax = plt.subplots(figsize=(10, 5))
        pivot = holdout.pivot_table(index="test_domain", columns="model", values="balanced_accuracy", aggfunc="mean")
        pivot.plot(kind="bar", ax=ax, rot=15)
        ax.set_title("Domain Holdout — Balanced Accuracy by Test Domain × Model", fontsize=14)
        ax.set_ylabel("Balanced Accuracy")
        ax.set_xlabel("Held-out Domain")
        ax.legend(title="Model")
        ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/domain_holdout_balanced_accuracy.png", dpi=DPI)
        plt.show()
        plt.close()  # release figure memory

# ── 2. Pooled CV: Box plots ─────────────────────────────────────────────
if CONFIG["pooled_cv"]:
    cv_data = metrics_df[metrics_df["run_type"] == "pooled_cv"].copy()
    if len(cv_data) > 0:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=cv_data, x="model", y="macro_f1", ax=ax, palette="Set2")
        ax.set_title("Pooled CV — Macro F1 Distribution by Model", fontsize=14)
        ax.set_ylabel("Macro F1")
        ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/pooled_cv_macro_f1_box.png", dpi=DPI)
        plt.show()
        plt.close()  # release figure memory

        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=cv_data, x="model", y="balanced_accuracy", ax=ax, palette="Set2")
        ax.set_title("Pooled CV — Balanced Accuracy Distribution by Model", fontsize=14)
        ax.set_ylabel("Balanced Accuracy")
        ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/pooled_cv_balanced_accuracy_box.png", dpi=DPI)
        plt.show()
        plt.close()  # release figure memory

# ── 3. Labeler Kappa ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette("viridis", len(kappa_df))
bars = ax.bar(kappa_df["scope"], kappa_df["kappa"], color=colors)
ax.set_title("Cohen's Kappa — Labeler Agreement by Scope", fontsize=14)
ax.set_ylabel("Cohen's Kappa")
kappa_min = float(kappa_df["kappa"].min()) if len(kappa_df) else 0
ax.set_ylim(min(-1, kappa_min - 0.05), 1)
for bar, val in zip(bars, kappa_df["kappa"]):
    y_text = val + 0.03 if val >= 0 else val - 0.08
    ax.text(bar.get_x() + bar.get_width()/2, y_text,
            f"{val:.3f}", ha="center", fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{fig_dir}/labeler_kappa.png", dpi=DPI)
plt.show()
plt.close()  # release figure memory

# ── 4. Calls per domain ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
domain_counts = calls_df["domain"].value_counts()
domain_counts.plot(kind="bar", ax=ax, color=sns.color_palette("muted"))
ax.set_title("Call Counts per Domain", fontsize=14)
ax.set_ylabel("Number of Calls")
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{fig_dir}/calls_per_domain.png", dpi=DPI)
plt.show()
plt.close()  # release figure memory

# Use try/except -- profile is initialised to None in Data Engineering cell,
# so this guard works correctly regardless of execution order (no NameError).
try:
    _profile_ok = profile is not None and len(profile) > 0
except NameError:
    _profile_ok = False
if _profile_ok:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(profile["domain"], profile["dup_rate"], color=sns.color_palette("rocket", len(profile)))
    ax.set_title("Within-Domain Duplicate Rate", fontsize=14)
    ax.set_ylabel("Duplicate Rate")
    ax.set_ylim(0, max(profile["dup_rate"].max() * 1.3, 0.1))
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(f"{fig_dir}/dup_rate_per_domain.png", dpi=DPI)
    plt.show()
    plt.close()  # release figure memory
else:
    print("⚠ Skipping Figure 5 (profile data not available).")

# ── 6. Class distribution (stacked bar) ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
dist_plot = labels_df.groupby(["domain", "label_final"]).size().unstack(fill_value=0)
dist_plot = dist_plot.reindex(columns=CLASS_ORDER, fill_value=0)
dist_plot.plot(kind="bar", stacked=True, ax=ax,
               color=["#e74c3c", "#f39c12", "#2ecc71"])
ax.set_title("Label Distribution per Domain", fontsize=14)
ax.set_ylabel("Count")
ax.set_xlabel("")
ax.legend(title="Label")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{fig_dir}/class_distribution.png", dpi=DPI)
plt.show()
plt.close()  # release figure memory

print(f"\n✓ All figures saved to {fig_dir}/")

# -- 7. Golden Evaluation: Model Performance vs Human Labels ---------------
golden_eval_data = metrics_df[metrics_df["run_type"] == "golden_eval"].copy()
if len(golden_eval_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Macro F1
    ax = axes[0]
    sns.barplot(data=golden_eval_data, x="model", y="macro_f1", ax=ax, palette="Set2")
    ax.set_title("Golden Evaluation -- Macro F1 vs Human Labels", fontsize=14)
    ax.set_ylabel("Macro F1")
    ax.set_ylim(0, 1)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=9)

    # Balanced Accuracy
    ax = axes[1]
    sns.barplot(data=golden_eval_data, x="model", y="balanced_accuracy", ax=ax, palette="Set2")
    ax.set_title("Golden Evaluation -- Balanced Accuracy vs Human Labels", fontsize=14)
    ax.set_ylabel("Balanced Accuracy")
    ax.set_ylim(0, 1)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=9)

    plt.tight_layout()
    plt.savefig(f"{fig_dir}/golden_eval_performance.png", dpi=DPI)
    plt.show()
    plt.close()  # release figure memory

# -- 8. Labeler Comparison: Agreement with Human Labels --------------------
if golden_cfg["enabled"] and golden_kappa_df is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#e74c3c", "#3498db", "#e67e22", "#2ecc71"]
    bars = ax.bar(golden_kappa_df["pair"], golden_kappa_df["kappa"], color=colors[:len(golden_kappa_df)])
    ax.set_title("Cohen's Kappa -- Human vs Synthetic Labelers", fontsize=14)
    ax.set_ylabel("Cohen's Kappa")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    gk_min = float(golden_kappa_df["kappa"].min()) if len(golden_kappa_df) else 0
    ax.set_ylim(min(-1, gk_min - 0.05), 1)
    for bar, val in zip(bars, golden_kappa_df["kappa"]):
        y_text = val + 0.03 if val >= 0 else val - 0.08
        ax.text(bar.get_x() + bar.get_width()/2, y_text,
                f"{val:.4f}", ha="center", fontsize=10)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(f"{fig_dir}/golden_kappa_comparison.png", dpi=DPI)
    plt.show()
    plt.close()  # release figure memory

print(f"\nAll figures saved to {fig_dir}/")


## § Academic Report

Auto-generated Markdown report with all results, tables, and figure references.  
Displayed inline and saved to `outputs/reports/report_v1.md`.

In [ ]:
# ── Academic Report Generation ───────────────────────────────────────────
from IPython.display import display, Markdown

# Build metric summary strings
holdout_table = ""
if CONFIG.get("domain_holdout"):
    h = metrics_df[metrics_df["run_type"] == "domain_holdout"]
    if len(h) > 0:
        ht = h.pivot_table(index="test_domain", columns="model",
                           values=["macro_f1", "balanced_accuracy"], aggfunc="mean")
        holdout_table = ht.to_markdown()

cv_table = ""
if CONFIG.get("pooled_cv"):
    c = metrics_df[metrics_df["run_type"] == "pooled_cv"]
    if len(c) > 0:
        cs = c.groupby("model").agg(
            macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
            bal_acc_mean=("balanced_accuracy", "mean"), bal_acc_std=("balanced_accuracy", "std"),
        ).round(4)
        cv_table = cs.to_markdown()

brier_table = ""
if CONFIG.get("probability_metrics") and "brier_score_avg" in metrics_df.columns:
    brier_data = metrics_df[metrics_df["brier_score_avg"].notna()]
    if len(brier_data) > 0:
        bs = brier_data.groupby("model")["brier_score_avg"].agg(["mean", "std"]).round(4)
        brier_table = bs.to_markdown()

# Domain profile table
# Use try/except for profile guard.
try:
    _profile_ok_rpt = profile is not None and len(profile) > 0
except NameError:
    _profile_ok_rpt = False
profile_md = profile.to_markdown(index=False) if _profile_ok_rpt else "*No domain profile data available.*"
kappa_md = kappa_df.to_markdown(index=False)

report = f"""# NPS-Proxy Prediction from Healthcare Call-Center Transcripts
## A Text-First Machine Learning Approach Using Synthetic Labels

---

## 1. Abstract

This study develops a text-first machine learning pipeline for predicting NPS-proxy classes (detractor, passive, promoter) from healthcare call-center transcripts. Synthetic-label experiments measure reproduction of deterministic proxy labels; human-label performance is reported separately using the golden dataset. Structured cue features are retained for ablation but labeler cue columns are excluded from model structured features to reduce circularity.

---

## 2. Data Description

**Source**: AIxBlock/92k-real-world-call-center-scripts-english (HuggingFace)

**Quality filters applied**:
- Minimum word count: {CONFIG['min_words']}
- SHA1-based deduplication tracking
- Sanitized call-level export available without transcript text

### Per-Domain Profile

{profile_md}

---

## 3. Feature Engineering

### Track A — Text Representation
- TF-IDF vectorization (max_features={CONFIG['tfidf_max_features']}, ngram_range={CONFIG['tfidf_ngram_range']}, min_df={CONFIG['tfidf_min_df']}, max_df={CONFIG['tfidf_max_df']})
- Dimensionality reduction via TruncatedSVD (n_components={CONFIG['svd_components']})

### Track B — Structured Features
Lexical features (n_chars, n_words_text, punctuation counts), speaker-tag presence, ASR/duration features (audio_duration_sec, audio_duration_valid, asr_confidence, word-confidence percentiles, wpm), and cue counts for labeling/audit. Labeler cue columns are excluded from model structured features by default.

### Scaling Strategy
For linear models (LogReg, LinearSVC), a ColumnTransformer applies {CONFIG['track_b_scaler']} to Track B features only, leaving Track A (TF-IDF/SVD) unscaled.

**Run mode**: `{run_mode}`
**Synthetic training target**: `{target_col}`

---

## 4. Synthetic Labeling

> **Important limitation**: Labels are synthetic approximations of customer satisfaction, NOT true NPS scores.

**Labeler A (Rule/Rubric)**: Scores based on weighted keyword counts (escalation: -2, unresolved: -1, thanks: +0.5, resolution: +1, apology: -0.5). Mapped to classes via thresholds (≤-1.5: detractor, ≥2.0: promoter).

**Labeler B (Evidence-Normalized Sentiment)**: Deterministic lexicon-based sentiment normalized by sentiment evidence, with bounded interaction cue adjustments. Mapped via thresholds (<={B_DETRACTOR_THRESHOLD}: detractor, >={B_PROMOTER_THRESHOLD}: promoter).

**Primary label**: consensus/adjudicated synthetic label (label_final). Disagreements tracked for analysis.

### Two-Way Labeler Agreement (Cohen's Kappa)

{kappa_md}

---

## 5. Modeling

| Model | Key Parameters |
|---|---|
| LogisticRegression | max_iter=3000, class_weight=balanced |
| LinearSVC | class_weight=balanced; calibration disabled by default for speed/sample-weight support |
| LightGBM | n_estimators=800, lr=0.05, num_leaves=63, class_weight=balanced |

All models use random_state={SEED} for reproducibility.

---

## 6. Synthetic-Label Domain Holdout Results

{holdout_table}

---

## 7. Cross-Validation Results (mean ± std)

{cv_table}

---

## 8. Calibration Analysis (Brier Scores)

{brier_table if brier_table else "No probability metrics available for this run."}

---

## 9. Limitations

1. **Synthetic-label evaluation is not true NPS performance**: Domain holdout and pooled CV primarily measure reproduction of deterministic proxy labels.
2. **Residual label-signal risk**: Even text-only synthetic-label models inherit the proxy labeler assumptions; the golden human-label CV is the cleaner performance estimate.
3. **Golden dataset size and reuse**: Human-reviewed labels are a limited subset and should ideally be split into development and final test sets for publication.
4. **Identifier and deduplication assumptions**: Internal IDs are domain-qualified and exact transcript hashes are audited, but near-duplicates are not detected.
5. **Domain scope**: Limited to 3 healthcare sub-domains.
6. **No audio features**: Only transcript text and metadata are used.

---

## 10. Reproducibility

- Configuration: CONFIG dictionary in Cell 2
- Random seed: {SEED}
- Dataset: AIxBlock/92k-real-world-call-center-scripts-english (HuggingFace)
"""

# Display inline
display(Markdown(report))

# Save to file
report_path = f"{CONFIG['outputs_dir']}/reports/report_v1.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)
print(f"\n✓ Report saved to {report_path}")


In [ ]:
# ── Save & Export ─────────────────────────────────────────────────────────

# Download outputs as a zip; no Google Drive mount required.
try:
    zip_path = f"{CONFIG['project_root']}/nps_outputs"
    shutil.make_archive(zip_path, 'zip', CONFIG["outputs_dir"])
    print(f"✓ Zip created: {zip_path}.zip")

    from google.colab import files
    files.download(f"{zip_path}.zip")
except Exception as e:
    print(f"⚠ Download skipped: {e}")

print("\n✓ Pipeline complete!")
